# 🎥 CCTV Footage Analysis — find what matters without watching hours of video

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OfficialAshish/home-camera-analysis/blob/main/cctv-analysis.ipynb)

This notebook scans every recording in a Google Drive folder and produces a **reviewable, cited index of everything that happened**:

| What you get | Where |
|---|---|
| Every **motion event**: start/end clock time, the video file, and offset inside the video (HH:MM:SS) | `CCTV_Report.xlsx` → *Events* |
| **Snapshots** (annotated + raw), a contact sheet and a short **clip** for each event | `events/<camera>/<date>/<event-id>/` |
| **People & objects** tracked (YOLO26 + ByteTrack): confidence, how long they stayed, speed, restricted zones | *People & Objects* |
| **Faces**: detected, quality-scored, cropped and **grouped into identities across days** (P0001, P0002 …); known staff can be labelled | *Faces*, *Face Identities* |
| **Suspicion score 0–100** with plain-English reasons: after-hours activity, camera tampering, loitering, running, crowding, hidden face, hands up, restricted zone, weapon-like objects, AI scene match | *Events* (sortable) |
| **AI description** of each flagged event (Qwen3-VL vision-language model) | *Events* → *AI description* |
| Activity **heatmaps**, timeline charts, an interactive **HTML dashboard** and a **highlight reel** of the top events | output folder |
| **Search**: free text (*"man in a helmet"*) or **by photo** (find every visit of a person) | cells at the end |

**How to use:** `Runtime → Change runtime type → T4 GPU`, check the settings in **Step 3**, then `Runtime → Run all`.
Re-running is safe and incremental: finished videos are skipped and reports always cover everything analysed so far.

## Step 1 · GPU check & Google Drive

In [ ]:
#@title Check GPU and mount Google Drive
import subprocess, os
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout or "⚠️ No GPU – use Runtime › Change runtime type › T4 GPU (CPU works, but slowly)")
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Colab – using local paths.")

## Step 2 · Install dependencies (~1–2 min)

In [ ]:
#@title Install packages
!pip -q install -U "ultralytics>=8.3.200" lap xlsxwriter "transformers>=4.57" accelerate sentencepiece protobuf
!apt-get -qq install -y ffmpeg > /dev/null 2>&1 || true
import importlib, ultralytics, transformers, torch, cv2
print("ultralytics", ultralytics.__version__, "| transformers", transformers.__version__,
      "| torch", torch.__version__, "| opencv", cv2.__version__, "| CUDA", torch.cuda.is_available())

### Analysis code
The next cells write the `cctv_analysis` package to `/content` (same code as the GitHub repo, so the notebook is fully self-contained).
You don't need to read or edit them — collapse the section and move on.

In [ ]:
import os; os.makedirs('/content/cctv_analysis', exist_ok=True)

In [ ]:
%%writefile /content/cctv_analysis/__init__.py
"""CCTV footage analysis toolkit: motion events, people/object tracking, faces,
suspicious-activity scoring and reviewer-friendly Excel/HTML reports."""
__version__ = "1.0.0"

In [ ]:
%%writefile /content/cctv_analysis/config.py
"""All tunable settings live here. Every field has a sensible default, so a run
only needs `input_dir` and `output_dir`."""
from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field, fields
from typing import Dict, List, Optional

VIDEO_EXTENSIONS = [
    ".mp4", ".avi", ".mkv", ".mov", ".m4v", ".dav", ".h264", ".264", ".h265",
    ".265", ".hevc", ".ts", ".mts", ".m2ts", ".flv", ".asf", ".wmv", ".3gp",
    ".mpg", ".mpeg", ".webm", ".vob", ".ps", ".mpeg4",
]

# Zero-shot prompts used by SigLIP-2 to tag keyframes. Prompts in the
# "suspicious" group raise the suspicion score.
DEFAULT_SUSPICIOUS_PROMPTS = [
    "a person hiding a product inside their clothes or pocket",
    "a person putting merchandise into a bag without paying",
    "a person reaching into the cash register or cash drawer",
    "a person taking money from a counter",
    "two people fighting or pushing each other",
    "a person threatening someone with a knife or gun",
    "a person wearing a mask or helmet covering their face indoors",
    "a person breaking or damaging property",
    "a person covering or tampering with the security camera",
    "a person climbing over the counter",
    "a person lying on the floor",
    "a person looking around nervously while touching products",
]
DEFAULT_NORMAL_PROMPTS = [
    "an empty shop with no people",
    "a customer browsing products on a shelf",
    "a customer paying at the counter",
    "a shopkeeper standing behind the counter",
    "people talking calmly in a shop",
    "a person cleaning the floor",
    "a delivery person carrying boxes",
    "a street outside a shop with vehicles passing",
]

DEFAULT_VLM_PROMPT = (
    "You are a security analyst reviewing CCTV footage from a retail shop. "
    "The image is a grid of frames from one motion event, in time order "
    "(left to right, top to bottom). Describe briefly what the people are "
    "doing. Then decide if anything looks suspicious (theft, concealing items, "
    "taking cash, violence, vandalism, tampering with the camera, entering a "
    "restricted area). Answer ONLY with JSON of the form "
    '{"description": "...", "suspicious": true/false, "category": "none|theft|'
    'cash_handling|violence|vandalism|tampering|intrusion|other", '
    '"confidence": 0.0-1.0, "reason": "..."}'
)


@dataclass
class Config:
    # ---------------------------------------------------------------- paths
    input_dir: str = "/content/drive/MyDrive/CCTV"
    output_dir: str = "/content/drive/MyDrive/CCTV_Analysis"
    work_dir: str = "/content/cctv_work"          # fast local scratch (DB, temp)
    recursive: bool = True
    extensions: List[str] = field(default_factory=lambda: list(VIDEO_EXTENSIONS))
    include_glob: str = ""                       # e.g. "*ch01*" to limit files
    exclude_glob: str = ""
    max_videos: int = 0                          # 0 = no limit
    only_newer_than_days: float = 0              # 0 = all files
    reprocess: bool = False                      # ignore the "already done" cache
    max_runtime_minutes: float = 0               # stop cleanly (and still build reports) after this; 0 = no limit
    known_faces_dir: str = ""                    # folder/<Name>/*.jpg (staff, owner)

    # ------------------------------------------------------------ time info
    timezone: str = "Asia/Kolkata"
    # Order in which the recording start time is resolved.
    time_sources: List[str] = field(
        default_factory=lambda: ["filename", "metadata", "vlm_osd", "mtime"])
    # Extra regexes with named groups Y m d H M S (optional) tried before built-ins.
    filename_time_patterns: List[str] = field(default_factory=list)
    # Manual overrides: {"file name or relative path": "2026-10-11 09:30:00"}
    time_overrides: Dict[str, str] = field(default_factory=dict)
    camera_name_source: str = "auto"             # auto | folder | filename | none

    # ------------------------------------------------------------ shop rules
    open_hour: str = "09:00"                     # activity outside -> after-hours
    close_hour: str = "21:00"
    closed_weekdays: List[int] = field(default_factory=list)  # 0=Mon .. 6=Sun
    # Restricted zones per camera, normalised polygon coords (0..1).
    # {"*": {"cash_counter": [[0.6,0.5],[1,0.5],[1,1],[0.6,1]]}, "ch02": {...}}
    zones: Dict[str, Dict[str, List[List[float]]]] = field(default_factory=dict)
    # Areas to ignore for motion (timestamp overlay, swaying plants, TV screens).
    ignore_regions: Dict[str, List[List[List[float]]]] = field(default_factory=dict)
    auto_mask_osd: bool = True                   # auto-detect burnt-in clock text

    # ---------------------------------------------------------- motion pass
    motion_fps: float = 4.0
    motion_width: int = 384
    motion_sensitivity: float = 0.5              # 0 (least) .. 1 (most sensitive)
    min_motion_area: float = 0.0015              # fraction of frame
    global_change_ratio: float = 0.65            # > this = lighting change, ignore
    event_pre_roll: float = 2.0                  # seconds added before motion
    event_post_roll: float = 3.0
    event_merge_gap: float = 6.0                 # merge events closer than this
    event_min_duration: float = 1.0
    event_max_duration: float = 600.0            # split very long events
    detect_tamper: bool = True

    # -------------------------------------------------------- detection pass
    device: str = "auto"                         # auto | cuda | cpu
    hw_decode: str = "auto"                      # auto | cuda | none  (NVDEC video decoding on Colab GPUs)
    detector_model: str = "yolo26s.pt"           # falls back to yolo11s.pt
    detector_fallbacks: List[str] = field(default_factory=lambda: ["yolo11s.pt", "yolov8s.pt"])
    pose_model: str = "yolo26n-pose.pt"
    use_pose: bool = True
    tracker: str = "bytetrack.yaml"
    detect_fps: float = 6.0
    detect_width: int = 1280
    det_conf: float = 0.30
    det_imgsz: int = 960
    # COCO classes that matter in a shop (None = everything).
    keep_classes: Optional[List[str]] = field(default_factory=lambda: [
        "person", "bicycle", "car", "motorcycle", "bus", "truck", "dog", "cat",
        "backpack", "umbrella", "handbag", "suitcase", "bottle", "cup", "knife",
        "scissors", "baseball bat", "cell phone", "laptop", "book", "chair",
    ])
    require_person_for_event: bool = False       # drop motion with no person/vehicle/animal

    # ----------------------------------------------------------------- faces
    use_faces: bool = True
    face_score_threshold: float = 0.75
    face_min_size: int = 20                      # px, in detection-resolution frame
    face_match_threshold: float = 0.42           # SFace cosine similarity
    known_face_threshold: float = 0.45
    faces_per_track: int = 3
    face_min_quality: float = 0.30               # below this a face is kept as evidence but not used for identity

    # ------------------------------------------------- suspicious heuristics
    loiter_seconds: float = 90.0
    crowd_size: int = 4
    running_speed: float = 1.6                   # body-heights per second
    weapon_classes: List[str] = field(default_factory=lambda: ["knife", "scissors", "baseball bat"])
    hidden_face_seconds: float = 8.0
    suspicion_alert_threshold: float = 40.0      # score >= this = "REVIEW"

    # ------------------------------------------------------ semantic models
    use_semantic: bool = True
    semantic_model: str = "google/siglip2-base-patch16-224"
    suspicious_prompts: List[str] = field(default_factory=lambda: list(DEFAULT_SUSPICIOUS_PROMPTS))
    normal_prompts: List[str] = field(default_factory=lambda: list(DEFAULT_NORMAL_PROMPTS))
    semantic_alert_score: float = 0.10           # SigLIP sigmoid; also needs lift >= 0.6

    use_vlm: bool = True
    vlm_model: str = "Qwen/Qwen3-VL-2B-Instruct"
    vlm_scope: str = "flagged"                   # flagged | people | all
    vlm_max_events: int = 300
    vlm_prompt: str = DEFAULT_VLM_PROMPT
    vlm_max_new_tokens: int = 180

    # --------------------------------------------------------------- outputs
    snapshots_per_event: int = 6
    save_raw_snapshots: bool = True
    snapshot_jpeg_quality: int = 88
    save_clips: bool = True
    clip_scope: str = "people"                   # all | people | flagged
    clip_max_seconds: float = 180.0
    clip_width: int = 960
    excel_thumbnails: bool = True
    excel_max_thumbnails: int = 3000
    make_html: bool = True
    make_highlight_reel: bool = True
    highlight_max_events: int = 40
    drive_links: bool = True                     # resolve Google Drive file links in Colab

    @classmethod
    def from_dict(cls, d: dict) -> "Config":
        names = {f.name for f in fields(cls)}
        unknown = set(d) - names
        if unknown:
            raise ValueError(f"Unknown config keys: {sorted(unknown)}")
        return cls(**d)

    @classmethod
    def from_json(cls, path: str) -> "Config":
        with open(path, "r", encoding="utf-8") as fh:
            return cls.from_dict(json.load(fh))

    def to_dict(self) -> dict:
        return asdict(self)

    def save(self, path: str) -> None:
        with open(path, "w", encoding="utf-8") as fh:
            json.dump(self.to_dict(), fh, indent=2)

In [ ]:
%%writefile /content/cctv_analysis/video_io.py
"""Video discovery, probing, fast frame reading (ffmpeg pipe with OpenCV
fallback), recording-time inference, clip export and Google-Drive links."""
from __future__ import annotations

import datetime as dt
import fnmatch
import json
import os
import re
import shutil
import subprocess
import time
from typing import Iterator, List, Optional, Tuple

import cv2
import numpy as np

try:
    from zoneinfo import ZoneInfo
except ImportError:  # pragma: no cover
    ZoneInfo = None

FFMPEG = shutil.which("ffmpeg")
FFPROBE = shutil.which("ffprobe")
HW_DECODE = "auto"          # auto | cuda | none   (set from Config.hw_decode)
_HW_STATE = {}              # path-independent cache: {"cuda": True/False}


def _cuda_present() -> bool:
    return shutil.which("nvidia-smi") is not None or os.path.exists("/dev/nvidia0")


def _hw_args() -> List[str]:
    if HW_DECODE == "none" or not FFMPEG:
        return []
    if HW_DECODE == "auto" and not _cuda_present():
        return []
    if _HW_STATE.get("cuda") is False:
        return []
    return ["-hwaccel", "cuda"]

# --------------------------------------------------------------------- files

def find_videos(cfg) -> List[str]:
    exts = {e.lower() for e in cfg.extensions}
    out = []
    if os.path.isfile(cfg.input_dir):
        return [cfg.input_dir]
    cutoff = time.time() - cfg.only_newer_than_days * 86400 if cfg.only_newer_than_days else None
    for root, dirs, files in os.walk(cfg.input_dir):
        # never descend into our own output folder
        dirs[:] = sorted(d for d in dirs
                         if os.path.abspath(os.path.join(root, d)) != os.path.abspath(cfg.output_dir))
        for f in sorted(files):
            if os.path.splitext(f)[1].lower() not in exts:
                continue
            p = os.path.join(root, f)
            if cfg.include_glob and not fnmatch.fnmatch(f, cfg.include_glob):
                continue
            if cfg.exclude_glob and fnmatch.fnmatch(f, cfg.exclude_glob):
                continue
            try:
                st = os.stat(p)
            except OSError:
                continue
            if st.st_size < 10_000:
                continue
            if cutoff and st.st_mtime < cutoff:
                continue
            out.append(p)
        if not cfg.recursive:
            break
    if cfg.max_videos:
        out = out[: cfg.max_videos]
    return out


def file_fingerprint(path: str) -> str:
    st = os.stat(path)
    return f"{st.st_size}-{int(st.st_mtime)}"

# ------------------------------------------------------------------- probing

def _input_args(path: str) -> List[str]:
    ext = os.path.splitext(path)[1].lower()
    if ext == ".dav":
        return ["-f", "dhav"]          # Dahua container
    if ext in (".h264", ".264"):
        return ["-f", "h264"]
    if ext in (".h265", ".265", ".hevc"):
        return ["-f", "hevc"]
    return []


def probe(path: str) -> dict:
    info = {"duration": 0.0, "fps": 0.0, "width": 0, "height": 0, "codec": "",
            "creation_time": None, "nb_frames": 0, "format": ""}
    if FFPROBE:
        for extra in (_input_args(path), []):
            try:
                r = subprocess.run(
                    [FFPROBE, "-v", "error", *extra, "-print_format", "json",
                     "-show_format", "-show_streams", path],
                    capture_output=True, text=True, timeout=120)
                j = json.loads(r.stdout or "{}")
            except Exception:
                j = {}
            vs = [s for s in j.get("streams", []) if s.get("codec_type") == "video"]
            if not vs:
                continue
            s = vs[0]
            fmt = j.get("format", {})
            info["codec"] = s.get("codec_name", "")
            info["width"] = int(s.get("width") or 0)
            info["height"] = int(s.get("height") or 0)
            info["format"] = fmt.get("format_name", "")
            for key in ("avg_frame_rate", "r_frame_rate"):
                try:
                    n, d = s.get(key, "0/1").split("/")
                    v = float(n) / float(d) if float(d) else 0
                    if 0 < v < 241:
                        info["fps"] = v
                        break
                except Exception:
                    pass
            for src in (fmt, s):
                try:
                    if float(src.get("duration") or 0) > 0:
                        info["duration"] = float(src["duration"])
                        break
                except Exception:
                    pass
            info["nb_frames"] = int(s.get("nb_frames") or 0)
            tags = {**fmt.get("tags", {}), **s.get("tags", {})}
            info["creation_time"] = tags.get("creation_time")
            break
    if not info["width"] or not info["duration"]:
        cap = cv2.VideoCapture(path)
        if cap.isOpened():
            info["width"] = info["width"] or int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            info["height"] = info["height"] or int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            fps = cap.get(cv2.CAP_PROP_FPS) or 0
            n = cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0
            info["fps"] = info["fps"] or (fps if 0 < fps < 241 else 0)
            if not info["duration"] and fps and n > 0:
                info["duration"] = n / fps
        cap.release()
    if not info["duration"] and FFMPEG:
        # last resort: decode-count (slow but reliable for raw streams)
        try:
            r = subprocess.run([FFMPEG, "-v", "error", *_input_args(path), "-i", path,
                                "-map", "0:v:0", "-c", "copy", "-f", "null", "-"],
                               capture_output=True, text=True, timeout=1800)
            m = re.findall(r"time=(\d+):(\d+):([\d.]+)", r.stderr)
            if m:
                h, mi, s = m[-1]
                info["duration"] = int(h) * 3600 + int(mi) * 60 + float(s)
        except Exception:
            pass
    return info

# ------------------------------------------------------------- frame reading

def iter_frames(path: str, fps: float, width: int, start: float = 0.0,
                duration: Optional[float] = None, gray: bool = False,
                src_size: Tuple[int, int] = (0, 0)) -> Iterator[Tuple[float, np.ndarray]]:
    """Yield (timestamp_seconds, frame) sampled at `fps`, resized to `width`."""
    sw, sh = src_size
    if width and sw and width >= sw:
        width = sw
    produced = False
    if FFMPEG and sw and sh:
        w = int(width or sw) // 2 * 2
        h = int(round(sh * w / sw)) // 2 * 2
        ch = 1 if gray else 3
        hw = _hw_args()
        cmd = [FFMPEG, "-nostdin", "-v", "error", *hw, "-threads", "0"]
        if start > 0:
            cmd += ["-ss", f"{start:.3f}"]
        cmd += [*_input_args(path), "-i", path]
        if duration:
            cmd += ["-t", f"{duration:.3f}"]
        cmd += ["-map", "0:v:0", "-an", "-sn", "-vf",
                f"fps={fps},scale={w}:{h}:flags=area", "-pix_fmt",
                "gray" if gray else "bgr24", "-f", "rawvideo", "pipe:1"]
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
                                bufsize=w * h * ch * 4)
        nbytes = w * h * ch
        i = 0
        try:
            while True:
                buf = proc.stdout.read(nbytes)
                if len(buf) < nbytes:
                    if i == 0 and hw:
                        proc.kill()
                        prev = _HW_STATE.get("cuda")
                        _HW_STATE["cuda"] = False      # retry on CPU
                        got = 0
                        for item in iter_frames(path, fps, width, start, duration, gray, src_size):
                            got += 1
                            yield item
                        if not got:                    # empty range, not a GPU problem
                            if prev is None:
                                _HW_STATE.pop("cuda", None)
                            else:
                                _HW_STATE["cuda"] = prev
                        return
                    break
                if i == 0 and hw:
                    _HW_STATE["cuda"] = True
                frame = np.frombuffer(buf, np.uint8).reshape((h, w) if gray else (h, w, 3))
                produced = True
                yield start + i / fps, frame
                i += 1
        finally:
            proc.stdout.close()
            proc.kill()
            proc.wait()
    if produced:
        return
    # ---- OpenCV fallback
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        return
    nfps = cap.get(cv2.CAP_PROP_FPS) or 25
    if not (0 < nfps < 241):
        nfps = 25
    if start > 0:
        cap.set(cv2.CAP_PROP_POS_MSEC, start * 1000)
    step = max(1, int(round(nfps / fps)))
    idx = 0
    end = start + duration if duration else None
    while True:
        ok = cap.grab()
        if not ok:
            break
        t = start + idx / nfps
        if end and t > end:
            break
        if idx % step == 0:
            ok, frame = cap.retrieve()
            if not ok:
                break
            if width and frame.shape[1] != width:
                h = int(round(frame.shape[0] * width / frame.shape[1]))
                frame = cv2.resize(frame, (width, h), interpolation=cv2.INTER_AREA)
            if gray:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            yield t, frame
        idx += 1
    cap.release()


def read_frame_at(path: str, t: float, width: int = 0, src_size=(0, 0)) -> Optional[np.ndarray]:
    for _, f in iter_frames(path, 1.0, width, start=max(0.0, t), duration=1.0, src_size=src_size):
        return f.copy()
    return None

# ------------------------------------------------------------ time inference

_BUILTIN_PATTERNS = [
    # 20261011143000, 2026-10-11_14-30-00, 2026.10.11 14.30.00, 20261011T143000
    r"(?P<Y>20\d{2})[-_.]?(?P<m>[01]\d)[-_.]?(?P<d>[0-3]\d)[-_ T.]?(?P<H>[0-2]\d)[-_.:hH]?(?P<M>[0-5]\d)[-_.:mM]?(?P<S>[0-5]\d)",
    # 11-10-2026_14-30-00 (day first)
    r"(?P<d>[0-3]\d)[-_.](?P<m>[01]\d)[-_.](?P<Y>20\d{2})[-_ T.]?(?P<H>[0-2]\d)[-_.:]?(?P<M>[0-5]\d)[-_.:]?(?P<S>[0-5]\d)",
    # date only + hour-minute
    r"(?P<Y>20\d{2})[-_.]?(?P<m>[01]\d)[-_.]?(?P<d>[0-3]\d)[-_ T.]?(?P<H>[0-2]\d)[-_.:]?(?P<M>[0-5]\d)(?!\d)",
]
_DATE_ONLY = r"(?P<Y>20\d{2})[-_.]?(?P<m>[01]\d)[-_.]?(?P<d>[0-3]\d)(?!\d)"
_TIME_ONLY = r"(?<!\d)(?P<H>[0-2]\d)[.:_-](?P<M>[0-5]\d)[.:_-](?P<S>[0-5]\d)"   # Dahua 14.30.00-14.45.00


def _tz(name: str):
    if ZoneInfo is None:
        return dt.timezone.utc
    try:
        return ZoneInfo(name)
    except Exception:
        return dt.timezone.utc


def _mk(g: dict, tz) -> Optional[dt.datetime]:
    try:
        d = dt.datetime(int(g["Y"]), int(g["m"]), int(g["d"]), int(g.get("H") or 0),
                        int(g.get("M") or 0), int(g.get("S") or 0), tzinfo=tz)
        if 2000 <= d.year <= 2100:
            return d
    except (ValueError, TypeError, KeyError):
        pass
    return None


def time_from_name(path: str, root: str, patterns: List[str], tz) -> Optional[dt.datetime]:
    name = os.path.basename(path)
    rel = os.path.relpath(path, root) if root and os.path.isdir(root) else path
    for pat in list(patterns) + _BUILTIN_PATTERNS:
        for text in (name, rel):
            m = re.search(pat, text)
            if m:
                d = _mk(m.groupdict(), tz)
                if d:
                    return d
    # date somewhere in path + time in file name (Dahua / many NVR exports)
    md = re.search(_DATE_ONLY, rel)
    mt = re.search(_TIME_ONLY, name)
    if md and mt:
        return _mk({**md.groupdict(), **mt.groupdict()}, tz)
    return None


def infer_start_time(path: str, meta: dict, cfg, vlm_reader=None) -> Tuple[Optional[dt.datetime], str]:
    """Return (aware datetime of first frame, source label)."""
    tz = _tz(cfg.timezone)
    base = os.path.basename(path)
    rel = os.path.relpath(path, cfg.input_dir) if os.path.isdir(cfg.input_dir) else base
    for key in (rel, base):
        if key in cfg.time_overrides:
            return dt.datetime.fromisoformat(cfg.time_overrides[key]).replace(tzinfo=tz), "override"
    for src in cfg.time_sources:
        if src == "filename":
            d = time_from_name(path, cfg.input_dir, cfg.filename_time_patterns, tz)
            if d:
                return d, "filename"
        elif src == "metadata" and meta.get("creation_time"):
            try:
                d = dt.datetime.fromisoformat(meta["creation_time"].replace("Z", "+00:00"))
                if d.tzinfo is None:
                    d = d.replace(tzinfo=dt.timezone.utc)
                if 2000 <= d.year <= 2100 and d.year != 1970:
                    return d.astimezone(tz), "metadata"
            except Exception:
                pass
        elif src == "vlm_osd" and vlm_reader is not None:
            try:
                d = vlm_reader(path, meta)
                if d:
                    return d.replace(tzinfo=tz) if d.tzinfo is None else d, "osd(vlm)"
            except Exception:
                pass
        elif src == "mtime":
            st = os.stat(path)
            d = dt.datetime.fromtimestamp(st.st_mtime, tz) - dt.timedelta(seconds=meta.get("duration") or 0)
            return d, "file-modified(approx)"
    return None, "unknown"


def camera_name(path: str, cfg) -> str:
    if cfg.camera_name_source == "none":
        return ""
    base = os.path.basename(path)
    m = re.search(r"(?i)(?:^|[^a-z])((?:ch|cam|camera|channel|d)[ _-]?\d{1,3})(?![0-9])", base)
    if cfg.camera_name_source in ("auto", "filename") and m:
        return re.sub(r"[ _-]", "", m.group(1)).lower()
    if cfg.camera_name_source in ("auto", "folder"):
        parent = os.path.dirname(os.path.abspath(path))
        if os.path.abspath(parent) != os.path.abspath(cfg.input_dir):
            rel = os.path.relpath(parent, cfg.input_dir)
            first = rel.split(os.sep)[0]
            if first and first != ".." and not re.fullmatch(r"[\d\-_.]+", first):
                return first
    return "cam"

# ---------------------------------------------------------------- clip export

def export_clip(path: str, start: float, duration: float, out_path: str,
                width: int = 960, label: str = "") -> bool:
    if not FFMPEG:
        return False
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    vf = f"scale='min({width},iw)':-2"
    if label:
        safe = re.sub(r"[^A-Za-z0-9 _:./-]", "", label).replace(":", r"\:")
        vf += (f",drawtext=text='{safe}':x=10:y=h-th-10:fontsize=18:fontcolor=white:"
               "box=1:boxcolor=black@0.5")
    base = [FFMPEG, "-nostdin", "-y", "-v", "error", "-ss", f"{max(0, start):.3f}",
            *_input_args(path), "-i", path, "-t", f"{duration:.3f}", "-map", "0:v:0", "-an"]
    tail = ["-c:v", "libx264", "-preset", "veryfast", "-crf", "27", "-pix_fmt", "yuv420p",
            "-movflags", "+faststart", out_path]
    for filt in (vf, f"scale='min({width},iw)':-2"):     # retry without drawtext
        try:
            r = subprocess.run(base + ["-vf", filt] + tail, capture_output=True, timeout=900)
            if r.returncode == 0 and os.path.exists(out_path) and os.path.getsize(out_path) > 1000:
                return True
        except Exception:
            pass
    return False

# --------------------------------------------------------------- drive links

def drive_link(path: str) -> str:
    """Google-Drive web link for a file on a Colab-mounted Drive ('' if unknown)."""
    try:
        fid = os.getxattr(path, "user.drive.id").decode()
        if fid:
            return f"https://drive.google.com/file/d/{fid}/view"
    except Exception:
        pass
    return ""


def fmt_offset(sec: float) -> str:
    sec = max(0.0, float(sec))
    h, rem = divmod(int(sec), 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

In [ ]:
%%writefile /content/cctv_analysis/motion.py
"""Pass 1: cheap full-video motion scan.

* MOG2 background subtraction on a small grey stream (default 384px @ 4 fps)
* automatic masking of the burnt-in DVR clock (OSD) + user ignore regions
* rejection of global lighting changes (lights on/off, IR switch, auto exposure)
* camera tamper detection (covered lens, blackout, defocus, moved camera)
* per-video activity heatmap
* motion intervals -> padded, merged, length-limited events
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import List, Optional, Tuple

import cv2
import numpy as np

from .video_io import iter_frames


@dataclass
class MotionResult:
    events: List[dict] = field(default_factory=list)
    times: np.ndarray = None
    ratios: np.ndarray = None
    heatmap: Optional[np.ndarray] = None
    tamper: List[dict] = field(default_factory=list)
    lighting_changes: List[float] = field(default_factory=list)
    osd_mask: Optional[np.ndarray] = None
    preview: Optional[np.ndarray] = None
    frames: int = 0


def poly_mask(shape: Tuple[int, int], polys, value=255) -> np.ndarray:
    h, w = shape
    m = np.zeros((h, w), np.uint8)
    for poly in polys or []:
        pts = np.array([[int(x * w), int(y * h)] for x, y in poly], np.int32)
        if len(pts) >= 3:
            cv2.fillPoly(m, [pts], value)
    return m


def detect_osd_mask(path: str, meta: dict, width: int, seconds: float = 90.0) -> Optional[np.ndarray]:
    """Find the DVR clock: small text-like regions near the frame border that change
    roughly once per second in an otherwise static picture."""
    frames = [f for _, f in iter_frames(path, 2.0, width, 0, min(seconds, meta.get("duration") or seconds),
                                        gray=True, src_size=(meta["width"], meta["height"]))]
    if len(frames) < 8:
        return None
    h, w = frames[0].shape
    freq = np.zeros((h, w), np.float32)
    for a, b in zip(frames, frames[1:]):
        freq += (cv2.absdiff(a, b) > 30)
    freq /= len(frames) - 1
    cand = (freq > 0.25).astype(np.uint8) * 255
    band = np.zeros_like(cand)
    band[: int(h * 0.18)] = 255
    band[int(h * 0.85):] = 255
    cand = cv2.bitwise_and(cand, band)
    cand = cv2.dilate(cand, cv2.getStructuringElement(cv2.MORPH_RECT, (15, 7)))
    n, lab, stats, _ = cv2.connectedComponentsWithStats(cand)
    mask = np.zeros((h, w), np.uint8)
    for i in range(1, n):
        x, y, bw, bh, area = stats[i]
        if bh < h * 0.12 and bw < w * 0.75 and area > 20:
            cv2.rectangle(mask, (max(0, x - 4), max(0, y - 4)), (min(w, x + bw + 4), min(h, y + bh + 4)), 255, -1)
    return mask if mask.any() else None


def _intervals(times, active, gap):
    out = []
    s = prev = None
    for t, a in zip(times, active):
        if not a:
            continue
        if s is None:
            s = prev = t
        elif t - prev > gap:
            out.append([s, prev])
            s = prev = t
        else:
            prev = t
    if s is not None:
        out.append([s, prev])
    return out


def scan_motion(path: str, meta: dict, cfg, camera: str = "") -> MotionResult:
    res = MotionResult()
    W, H = meta["width"], meta["height"]
    width = min(cfg.motion_width, W) if W else cfg.motion_width
    sens = float(np.clip(cfg.motion_sensitivity, 0, 1))
    var_thr = 12 + (1 - sens) * 44
    diff_thr = int(14 + (1 - sens) * 26)
    min_area = cfg.min_motion_area * (1.6 - sens)
    fps = cfg.motion_fps

    mog = cv2.createBackgroundSubtractorMOG2(history=int(fps * 120), varThreshold=var_thr,
                                             detectShadows=True)
    # only *mild* darkening counts as shadow; dark clothing must stay foreground
    mog.setShadowThreshold(0.82)
    kernel_o = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    kernel_d = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

    ign = None
    times, ratios, bright, sharp, glob = [], [], [], [], []
    heat = None
    prev = None
    for i, (t, g) in enumerate(iter_frames(path, fps, width, gray=True, src_size=(W, H))):
        if ign is None:
            h, w = g.shape
            polys = list(cfg.ignore_regions.get("*", [])) + list(cfg.ignore_regions.get(camera, []))
            ign = poly_mask((h, w), polys)
            if cfg.auto_mask_osd:
                osd = detect_osd_mask(path, meta, width)
                if osd is not None and osd.shape == ign.shape:
                    ign = cv2.bitwise_or(ign, osd)
                    res.osd_mask = osd
            keep = cv2.bitwise_not(ign)
            heat = np.zeros((h, w), np.float32)
            valid_px = max(1, int(np.count_nonzero(keep)))
        gb = cv2.GaussianBlur(g, (5, 5), 0)
        lr = 1.0 if i == 0 else -1
        fg = mog.apply(gb, learningRate=lr)
        fg = np.where(fg > 200, 255, 0).astype(np.uint8)          # drop shadows (127)
        if prev is not None:                                        # confirm with frame diff
            d = cv2.absdiff(gb, prev)
            dm = (d > diff_thr).astype(np.uint8) * 255
            dm = cv2.dilate(dm, kernel_d, iterations=2)
            fg = cv2.bitwise_and(fg, dm)
        prev = gb
        fg = cv2.bitwise_and(fg, keep)
        raw_ratio = np.count_nonzero(fg) / valid_px
        fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, kernel_o)
        fg = cv2.dilate(fg, kernel_d)
        n, _, stats, _ = cv2.connectedComponentsWithStats(fg)
        blob_min = valid_px * min_area * 0.25
        area = sum(int(stats[k, 4]) for k in range(1, n) if stats[k, 4] >= blob_min)
        ratio = area / valid_px
        is_global = raw_ratio > cfg.global_change_ratio
        if i < 3:
            ratio = 0.0
        if is_global:
            # lighting/exposure jump: rebuild the background from this frame
            if not res.lighting_changes or t - res.lighting_changes[-1] > 1.5:
                res.lighting_changes.append(float(t))
            mog.apply(gb, learningRate=1.0)
            ratio = 0.0
        else:
            heat += (fg > 0)
        times.append(t)
        ratios.append(ratio)
        glob.append(is_global)
        if cfg.detect_tamper and i % max(1, int(fps)) == 0:
            bright.append((t, float(g.mean()), float(cv2.Laplacian(g, cv2.CV_32F).var()), float(g.std())))
        if res.preview is None and i == int(fps * 2):
            res.preview = g.copy()
    res.frames = len(times)
    res.times = np.asarray(times, np.float32)
    res.ratios = np.asarray(ratios, np.float32)
    if heat is not None and heat.max() > 0:
        res.heatmap = heat / heat.max()
    if not times:
        return res

    # ---------------- events
    active = res.ratios >= min_area
    # require 2 active samples within a short window to suppress single-frame noise
    sm = np.convolve(active.astype(np.float32), np.ones(3), mode="same")
    active = active & (sm >= 2)
    ivs = _intervals(res.times, active, cfg.event_merge_gap)
    dur = float(meta.get("duration") or res.times[-1])
    for s, e in ivs:
        if e - s + 1.0 / fps < cfg.event_min_duration:
            continue
        s2, e2 = max(0.0, s - cfg.event_pre_roll), min(dur, e + cfg.event_post_roll)
        # split very long events
        cur = s2
        while cur < e2:
            nxt = min(e2, cur + cfg.event_max_duration)
            sel = (res.times >= cur) & (res.times <= nxt)
            r = res.ratios[sel]
            res.events.append({"start": float(cur), "end": float(nxt),
                               "motion_peak": float(r.max()) if r.size else 0.0,
                               "motion_mean": float(r.mean()) if r.size else 0.0,
                               "motion_active_s": float(np.count_nonzero(active[sel]) / fps)})
            cur = nxt
    # merge padded overlaps
    merged = []
    for ev in res.events:
        if merged and ev["start"] <= merged[-1]["end"] and \
                ev["end"] - merged[-1]["start"] <= cfg.event_max_duration:
            m = merged[-1]
            m["end"] = max(m["end"], ev["end"])
            m["motion_peak"] = max(m["motion_peak"], ev["motion_peak"])
            m["motion_mean"] = (m["motion_mean"] + ev["motion_mean"]) / 2
            m["motion_active_s"] += ev["motion_active_s"]
        else:
            merged.append(ev)
    res.events = merged

    # ---------------- tamper detection
    if cfg.detect_tamper and len(bright) >= 10:
        arr = np.array(bright)
        base_b = np.median(arr[:, 1])
        base_s = np.median(arr[:, 2])
        base_d = np.median(arr[:, 3])
        flags = []
        for t, b, s, d in arr:
            kind = None
            if b < 12 and base_b > 35:
                kind = "blackout/covered"
            elif s < 0.15 * base_s and d < 0.5 * base_d and d < 20 and base_d > 15:
                kind = "lens covered/obstructed"
            elif s < 0.08 * base_s and base_s > 50:
                kind = "defocus/blur"
            flags.append((t, kind))
        cur = None
        for t, kind in flags:
            if kind and cur and cur["kind"] == kind and t - cur["end"] <= 3:
                cur["end"] = float(t)
            elif kind:
                if cur:
                    res.tamper.append(cur)
                cur = {"start": float(t), "end": float(t), "kind": kind}
        if cur:
            res.tamper.append(cur)
        res.tamper = [x for x in res.tamper
                      if x["end"] - x["start"] >= (10 if x["kind"] == "defocus/blur" else 3)]
        # sustained global change = camera moved / scene changed
        g = np.array(glob)
        run = 0
        for t, gg in zip(res.times, g):
            run = run + 1 if gg else 0
            if run == int(fps * 4):
                res.tamper.append({"start": float(t - 4), "end": float(t), "kind": "scene change / camera moved"})
    return res

In [ ]:
%%writefile /content/cctv_analysis/detector.py
"""Pass 2 models: Ultralytics YOLO (detection + multi-object tracking) and an
optional pose model used for posture cues (crouching, hands raised, face
visibility)."""
from __future__ import annotations

import logging
from typing import List, Optional

import numpy as np

log = logging.getLogger("cctv")

COCO_KPT = {"nose": 0, "l_eye": 1, "r_eye": 2, "l_ear": 3, "r_ear": 4, "l_sh": 5, "r_sh": 6,
            "l_el": 7, "r_el": 8, "l_wr": 9, "r_wr": 10, "l_hip": 11, "r_hip": 12,
            "l_kn": 13, "r_kn": 14, "l_an": 15, "r_an": 16}


def resolve_device(pref: str = "auto") -> str:
    if pref and pref != "auto":
        return pref
    try:
        import torch
        if torch.cuda.is_available():
            return "cuda:0"
        if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
            return "mps"
    except Exception:
        pass
    return "cpu"


def load_yolo(candidates: List[str]):
    from ultralytics import YOLO
    last = None
    for name in candidates:
        if not name:
            continue
        try:
            m = YOLO(name)
            log.info("Loaded %s", name)
            return m, name
        except Exception as e:  # weights missing / unsupported by installed version
            last = e
            log.warning("Could not load %s (%s); trying fallback", name, e)
    raise RuntimeError(f"No YOLO model could be loaded: {last}")


def iou(a, b) -> float:
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0


class Detector:
    def __init__(self, cfg):
        self.cfg = cfg
        self.device = resolve_device(cfg.device)
        self.half = self.device.startswith("cuda")
        self.model, self.model_name = load_yolo([cfg.detector_model, *cfg.detector_fallbacks])
        names = self.model.names
        self.names = names if isinstance(names, dict) else dict(enumerate(names))
        if cfg.keep_classes:
            wanted = set(cfg.keep_classes)
            self.class_ids = [i for i, n in self.names.items() if n in wanted]
        else:
            self.class_ids = None
        self.pose = None
        self.pose_name = ""
        if cfg.use_pose and cfg.pose_model:
            try:
                self.pose, self.pose_name = load_yolo([cfg.pose_model, "yolo11n-pose.pt", "yolov8n-pose.pt"])
            except Exception as e:
                log.warning("Pose model disabled: %s", e)
        self._fresh = True
        self._extra = {"half": True} if self.half else {}

    def reset(self):
        """Start a new, independent tracking session (one per event)."""
        self._fresh = True

    def __call__(self, frame: np.ndarray) -> List[dict]:
        cfg = self.cfg
        r = self.model.track(frame, persist=not self._fresh, tracker=cfg.tracker, conf=cfg.det_conf,
                             imgsz=cfg.det_imgsz, classes=self.class_ids, device=self.device,
                             verbose=False, **self._extra)[0]
        self._fresh = False
        dets = []
        b = r.boxes
        if b is not None and len(b):
            xyxy = b.xyxy.cpu().numpy()
            conf = b.conf.cpu().numpy()
            cls = b.cls.cpu().numpy().astype(int)
            ids = b.id.cpu().numpy().astype(int) if b.id is not None else [None] * len(cls)
            for k in range(len(cls)):
                dets.append({"cls": self.names.get(int(cls[k]), str(cls[k])), "conf": float(conf[k]),
                             "xyxy": [float(v) for v in xyxy[k]],
                             "track": int(ids[k]) if ids[k] is not None else None, "kpts": None})
        if self.pose is not None and any(d["cls"] == "person" for d in dets):
            try:
                p = self.pose.predict(frame, conf=max(0.25, cfg.det_conf - 0.05), imgsz=cfg.det_imgsz,
                                      device=self.device, verbose=False, **self._extra)[0]
                if p.keypoints is not None and p.boxes is not None and len(p.boxes):
                    pb = p.boxes.xyxy.cpu().numpy()
                    kp = p.keypoints.data.cpu().numpy()          # (n,17,3)
                    for d in dets:
                        if d["cls"] != "person":
                            continue
                        best, bi = 0.0, -1
                        for j in range(len(pb)):
                            v = iou(d["xyxy"], pb[j])
                            if v > best:
                                best, bi = v, j
                        if best > 0.5:
                            d["kpts"] = kp[bi]
            except Exception as e:
                log.debug("pose failed: %s", e)
        return dets


# ------------------------------------------------------------ pose heuristics

def posture(kpts: Optional[np.ndarray], conf_thr: float = 0.4) -> dict:
    """Simple, explainable posture cues from COCO keypoints."""
    out = {"face_visible_pose": False, "crouching": False, "hands_up": False}
    if kpts is None:
        return out
    k = kpts

    def ok(n):
        return k[COCO_KPT[n], 2] >= conf_thr

    def p(n):
        return k[COCO_KPT[n], :2]

    out["face_visible_pose"] = ok("nose") and ok("l_eye") and ok("r_eye")
    if ok("l_sh") and ok("r_sh") and ok("l_hip") and ok("r_hip"):
        sh = (p("l_sh") + p("r_sh")) / 2
        hip = (p("l_hip") + p("r_hip")) / 2
        torso = np.linalg.norm(sh - hip) + 1e-6
        if ok("l_kn") and ok("r_kn"):
            kn = (p("l_kn") + p("r_kn")) / 2
            # knees close to hip height -> crouching / squatting / bending low
            if (kn[1] - hip[1]) < 0.35 * torso:
                out["crouching"] = True
        for w in ("l_wr", "r_wr"):
            if ok(w):
                wr = p(w)
                if wr[1] < sh[1] - 0.25 * torso:
                    out["hands_up"] = True
        if out["hands_up"]:
            both = ok("l_wr") and ok("r_wr") and p("l_wr")[1] < sh[1] and p("r_wr")[1] < sh[1]
            out["hands_up"] = bool(both)
    return out

In [ ]:
%%writefile /content/cctv_analysis/faces.py
"""Face detection (OpenCV YuNet), quality scoring, recognition embeddings
(OpenCV SFace, Apache-2.0) and a persistent cross-video identity database
with optional known-person enrolment (staff / owner)."""
from __future__ import annotations

import json
import logging
import os
import urllib.request
from typing import List, Optional

import cv2
import numpy as np

log = logging.getLogger("cctv")

CACHE = os.path.expanduser("~/.cache/cctv_analysis")
MODELS = {
    "yunet": ["https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx",
              "https://huggingface.co/opencv/face_detection_yunet/resolve/main/face_detection_yunet_2023mar.onnx"],
    "sface": ["https://github.com/opencv/opencv_zoo/raw/main/models/face_recognition_sface/face_recognition_sface_2021dec.onnx",
              "https://huggingface.co/opencv/face_recognition_sface/resolve/main/face_recognition_sface_2021dec.onnx"],
}


def fetch(key: str) -> str:
    os.makedirs(CACHE, exist_ok=True)
    dst = os.path.join(CACHE, os.path.basename(MODELS[key][0]))
    if os.path.exists(dst) and os.path.getsize(dst) > 100_000:
        return dst
    for url in MODELS[key]:
        try:
            urllib.request.urlretrieve(url, dst + ".part")
            if os.path.getsize(dst + ".part") > 100_000:
                os.replace(dst + ".part", dst)
                return dst
        except Exception as e:
            log.warning("download failed %s: %s", url, e)
    raise RuntimeError(f"could not download {key} model")


def sharpness(img: np.ndarray) -> float:
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    return float(cv2.Laplacian(g, cv2.CV_32F).var())


class FaceEngine:
    def __init__(self, cfg):
        self.cfg = cfg
        self.det = cv2.FaceDetectorYN.create(fetch("yunet"), "", (320, 320), cfg.face_score_threshold, 0.3, 50)
        self.rec = cv2.FaceRecognizerSF.create(fetch("sface"), "")

    def detect_in_people(self, frame: np.ndarray, persons: List[dict]) -> List[dict]:
        """Search faces in the upper part of each person box (much better than the
        full frame for small CCTV faces). Returns faces tagged with track id."""
        H, W = frame.shape[:2]
        out = []
        for p in persons:
            x1, y1, x2, y2 = p["xyxy"]
            bw, bh = x2 - x1, y2 - y1
            if bw < 12 or bh < 24:
                continue
            cx1 = int(max(0, x1 - 0.1 * bw)); cx2 = int(min(W, x2 + 0.1 * bw))
            # full body (tall box): face is in the top half; close-up / partial body: search almost all of it
            frac = 0.5 if bh / max(bw, 1) > 2.0 else 0.9
            cy1 = int(max(0, y1 - 0.08 * bh)); cy2 = int(min(H, y1 + frac * bh))
            crop = frame[cy1:cy2, cx1:cx2]
            if crop.size == 0:
                continue
            scale = 1.0
            if crop.shape[1] < 200:
                scale = 200.0 / crop.shape[1]
                crop = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
            self.det.setInputSize((crop.shape[1], crop.shape[0]))
            _, faces = self.det.detect(crop)
            if faces is None:
                continue
            for f in faces:
                f = f.copy()
                f[:14] /= scale
                # layout: x, y, w, h, 5 landmarks (x, y), score -> offset positions, not sizes
                f[[0, 4, 6, 8, 10, 12]] += cx1
                f[[1, 5, 7, 9, 11, 13]] += cy1
                fw, fh = f[2], f[3]
                if min(fw, fh) < self.cfg.face_min_size:
                    continue
                out.append({"track": p.get("track"), "row": f, "score": float(f[14]),
                            "box": [float(f[0]), float(f[1]), float(f[0] + fw), float(f[1] + fh)]})
                break  # one face per person
        return out

    def detect_full(self, frame: np.ndarray) -> List[dict]:
        self.det.setInputSize((frame.shape[1], frame.shape[0]))
        _, faces = self.det.detect(frame)
        res = []
        for f in faces if faces is not None else []:
            res.append({"track": None, "row": f, "score": float(f[14]),
                        "box": [float(f[0]), float(f[1]), float(f[0] + f[2]), float(f[1] + f[3])]})
        return res

    def quality(self, frame: np.ndarray, face: dict) -> float:
        f = face["row"]
        x, y, w, h = [int(v) for v in f[:4]]
        crop = frame[max(0, y):y + h, max(0, x):x + w]
        if crop.size == 0:
            return 0.0
        # frontal-ness from landmarks: nose centred between the eyes
        le, re_, nose = f[4:6], f[6:8], f[8:10]
        eye_d = abs(re_[0] - le[0]) + 1e-6
        frontal = 1.0 - min(1.0, abs((nose[0] - (le[0] + re_[0]) / 2) / eye_d) * 1.5)
        size = min(1.0, min(w, h) / 80.0)
        sharp = min(1.0, sharpness(crop) / 150.0)
        return float(face["score"] * (0.4 * size + 0.3 * sharp + 0.3 * max(0, frontal)))

    def embed(self, frame: np.ndarray, face: dict) -> Optional[np.ndarray]:
        try:
            aligned = self.rec.alignCrop(frame, face["row"])
            e = self.rec.feature(aligned).flatten().astype(np.float32)
            return e / (np.linalg.norm(e) + 1e-9)
        except Exception:
            return None

    def crop(self, frame: np.ndarray, box, pad: float = 0.35) -> np.ndarray:
        H, W = frame.shape[:2]
        x1, y1, x2, y2 = box
        w, h = x2 - x1, y2 - y1
        return frame[int(max(0, y1 - pad * h)):int(min(H, y2 + pad * h)),
                     int(max(0, x1 - pad * w)):int(min(W, x2 + pad * w))].copy()


class IdentityDB:
    """Greedy online clustering of face embeddings, persisted between runs so that
    'P0007' means the same person across days. Known people come from
    known_faces_dir/<Name>/*.jpg and are matched first."""

    def __init__(self, path: str, match_thr: float, known_thr: float):
        self.path = path
        self.thr = match_thr
        self.known_thr = known_thr
        self.ids: List[dict] = []
        self.emb: List[np.ndarray] = []
        self.known: List[dict] = []
        if os.path.exists(path + ".json") and os.path.exists(path + ".npy"):
            with open(path + ".json") as fh:
                self.ids = json.load(fh)
            arr = np.load(path + ".npy")
            self.emb = [a for a in arr]
            if len(self.emb) != len(self.ids):
                self.ids, self.emb = [], []

    def enroll_known(self, engine: FaceEngine, folder: str) -> int:
        if not folder or not os.path.isdir(folder):
            return 0
        n = 0
        for name in sorted(os.listdir(folder)):
            d = os.path.join(folder, name)
            files = [os.path.join(d, f) for f in sorted(os.listdir(d))] if os.path.isdir(d) else [d]
            label = name if os.path.isdir(d) else os.path.splitext(name)[0]
            embs = []
            for f in files:
                img = cv2.imread(f)
                if img is None:
                    continue
                faces = sorted(engine.detect_full(img), key=lambda x: -x["score"] * (x["box"][2] - x["box"][0]))
                if faces:
                    e = engine.embed(img, faces[0])
                    if e is not None:
                        embs.append(e)
            if embs:
                c = np.mean(embs, 0)
                self.known.append({"name": label, "emb": c / np.linalg.norm(c), "n": len(embs)})
                n += 1
        log.info("Enrolled %d known people", n)
        return n

    def assign(self, emb: np.ndarray, when: str, sample: str = "", event_id: str = "") -> tuple:
        """Return (identity_id, similarity, known_name)."""
        known_name = ""
        for k in self.known:
            s = float(np.dot(emb, k["emb"]))
            if s >= self.known_thr and (not known_name or s > known_sim):
                known_name, known_sim = k["name"], s
        best, bi = -1.0, -1
        if self.emb:
            sims = np.stack(self.emb) @ emb
            bi = int(np.argmax(sims))
            best = float(sims[bi])
        if bi >= 0 and best >= self.thr:
            rec = self.ids[bi]
            n = rec["count"]
            c = self.emb[bi] * n + emb
            self.emb[bi] = c / np.linalg.norm(c)
            rec["count"] = n + 1
            ev = rec.setdefault("events", [])
            if event_id and event_id not in ev:
                ev.append(event_id)
            rec["last_seen"] = max(rec["last_seen"], when)
            rec["first_seen"] = min(rec["first_seen"], when)
            if known_name and not rec.get("name"):
                rec["name"] = known_name
            return rec["id"], best, rec.get("name", "") or known_name
        new_id = f"P{len(self.ids) + 1:04d}"
        self.ids.append({"id": new_id, "count": 1, "first_seen": when, "last_seen": when,
                         "sample": sample, "name": known_name, "events": [event_id] if event_id else []})
        self.emb.append(emb)
        return new_id, 1.0, known_name

    def set_sample(self, pid: str, path: str):
        for r in self.ids:
            if r["id"] == pid and not r.get("sample"):
                r["sample"] = path

    def save(self):
        os.makedirs(os.path.dirname(self.path) or ".", exist_ok=True)
        with open(self.path + ".json", "w") as fh:
            json.dump(self.ids, fh, indent=1)
        np.save(self.path + ".npy", np.stack(self.emb) if self.emb else np.zeros((0, 128), np.float32))

In [ ]:
%%writefile /content/cctv_analysis/semantic.py
"""SigLIP-2 zero-shot scene tagging of keyframes + an embedding index that
powers free-text search ("person in red shirt near counter")."""
from __future__ import annotations

import logging
import os
from typing import List, Tuple

import numpy as np

log = logging.getLogger("cctv")


class Semantic:
    def __init__(self, cfg, device: str):
        import torch
        from transformers import AutoModel, AutoProcessor
        self.torch = torch
        self.device = device if device.startswith("cuda") else "cpu"
        dtype = torch.float16 if self.device.startswith("cuda") else torch.float32
        self.model = AutoModel.from_pretrained(cfg.semantic_model, dtype=dtype).to(self.device).eval()
        self.proc = AutoProcessor.from_pretrained(cfg.semantic_model)
        self.dtype = dtype
        self.sus = list(cfg.suspicious_prompts)
        self.norm = list(cfg.normal_prompts)
        self.prompts = self.sus + self.norm
        self.text_emb = self.encode_text([f"this is a photo of {x}." for x in self.prompts])
        self.scale = float(self.model.logit_scale.exp().item())
        self.bias = float(self.model.logit_bias.item()) if hasattr(self.model, "logit_bias") else 0.0

    def encode_text(self, texts: List[str]) -> np.ndarray:
        with self.torch.no_grad():
            t = self.proc(text=[x.lower() for x in texts], padding="max_length", max_length=64,
                          truncation=True, return_tensors="pt").to(self.device)
            e = self.model.get_text_features(**t)
            if not isinstance(e, self.torch.Tensor):
                e = getattr(e, "pooler_output", e[0])
            e = e.float()
            e = e / e.norm(dim=-1, keepdim=True)
        return e.cpu().numpy()

    def encode_images(self, images_bgr: List[np.ndarray], batch: int = 16) -> np.ndarray:
        from PIL import Image
        outs = []
        for i in range(0, len(images_bgr), batch):
            ims = [Image.fromarray(im[:, :, ::-1]) for im in images_bgr[i:i + batch]]
            with self.torch.no_grad():
                x = self.proc(images=ims, return_tensors="pt").to(self.device)
                x = {k: (v.to(self.dtype) if v.dtype.is_floating_point else v) for k, v in x.items()}
                e = self.model.get_image_features(**x)
                if not isinstance(e, self.torch.Tensor):
                    e = getattr(e, "pooler_output", e[0])
                e = e.float()
                e = e / e.norm(dim=-1, keepdim=True)
            outs.append(e.cpu().numpy())
        return np.concatenate(outs) if outs else np.zeros((0, self.text_emb.shape[1]), np.float32)

    def classify(self, img_emb: np.ndarray) -> List[dict]:
        """SigLIP's native per-prompt sigmoid scores (well calibrated: ~0 when no
        prompt fits). 'lift' compares the best suspicious vs best normal prompt."""
        res = []
        logits = img_emb @ self.text_emb.T * self.scale + self.bias
        sig = 1 / (1 + np.exp(-logits))
        ns = len(self.sus)
        for row in sig:
            top = int(np.argmax(row))
            ts = int(np.argmax(row[:ns]))
            tn = ns + int(np.argmax(row[ns:])) if len(row) > ns else None
            best_n = float(row[tn]) if tn is not None else 0.0
            lift = float(row[ts] / (row[ts] + best_n + 1e-9))
            res.append({"top_label": self.prompts[top], "top_score": float(row[top]),
                        "top_suspicious": self.sus[ts], "suspicious_score": float(row[ts]),
                        "top_normal": self.prompts[tn] if tn is not None else "", "normal_score": best_n,
                        "lift": lift})
        return res


def search_index(sem: Semantic, index_path: str, query: str, k: int = 20) -> List[Tuple[str, float]]:
    """Rank stored keyframes by similarity to a free-text query."""
    if not os.path.exists(index_path):
        return []
    z = np.load(index_path, allow_pickle=True)
    emb, keys = z["emb"].astype(np.float32), z["keys"]
    q = sem.encode_text([f"this is a photo of {query}."])[0]
    s = emb @ q
    order = np.argsort(-s)[:k]
    return [(str(keys[i]), float(s[i])) for i in order]

In [ ]:
%%writefile /content/cctv_analysis/vlm.py
"""Vision-language model (default Qwen3-VL-2B-Instruct) that writes a short
natural-language description of each event's contact sheet, gives its own
suspicious/not verdict, and can read the DVR's burnt-in clock (OSD)."""
from __future__ import annotations

import datetime as dt
import json
import logging
import re
from typing import Optional

import numpy as np

log = logging.getLogger("cctv")


class VLM:
    def __init__(self, cfg, device: str):
        import torch
        from transformers import AutoModelForImageTextToText, AutoProcessor
        self.torch = torch
        self.cfg = cfg
        cuda = device.startswith("cuda")
        if cuda:
            if torch.cuda.is_bf16_supported():
                dtype = torch.bfloat16
            else:
                # T4 & older: fp16 can overflow in Qwen-VL; small models fit comfortably in fp32
                small = any(t in cfg.vlm_model for t in ("256M", "500M", "0.5B", "1B", "2B"))
                dtype = torch.float32 if small else torch.float16
        else:
            dtype = torch.float32
        self.model = AutoModelForImageTextToText.from_pretrained(
            cfg.vlm_model, dtype=dtype, device_map="auto" if cuda else None).eval()
        if not cuda:
            self.model.to("cpu")
        self.proc = AutoProcessor.from_pretrained(cfg.vlm_model)

    def ask(self, img_bgr: np.ndarray, prompt: str, max_new_tokens: int = 160, max_side: int = 1280) -> str:
        from PIL import Image
        h, w = img_bgr.shape[:2]
        s = min(1.0, max_side / max(h, w))
        im = Image.fromarray(img_bgr[:, :, ::-1])
        if s < 1:
            im = im.resize((int(w * s), int(h * s)))
        messages = [{"role": "user", "content": [{"type": "image", "image": im},
                                                 {"type": "text", "text": prompt}]}]
        inputs = self.proc.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                               return_dict=True, return_tensors="pt")
        inputs = inputs.to(self.model.device)
        with self.torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        gen = out[:, inputs["input_ids"].shape[1]:]
        return self.proc.batch_decode(gen, skip_special_tokens=True)[0].strip()

    def describe_event(self, sheet_bgr: np.ndarray, context: str = "") -> dict:
        prompt = self.cfg.vlm_prompt + (f"\nContext from detectors: {context}" if context else "")
        txt = self.ask(sheet_bgr, prompt, self.cfg.vlm_max_new_tokens)
        return parse_verdict(txt)

    def read_clock(self, frame_bgr: np.ndarray) -> Optional[dt.datetime]:
        txt = self.ask(frame_bgr, "Read the date and time printed on this CCTV frame. Reply ONLY in the "
                                  "format YYYY-MM-DD HH:MM:SS, or NONE if there is no timestamp.", 40)
        return parse_clock(txt)


def parse_verdict(txt: str) -> dict:
    out = {"description": txt.strip(), "suspicious": None, "category": "", "confidence": None,
           "reason": "", "raw": txt}
    m = re.search(r"\{.*\}", txt, re.S)
    if m:
        raw = m.group(0)
        for cand in (raw, raw.replace("'", '"'), re.sub(r",\s*}", "}", raw)):
            try:
                j = json.loads(cand)
                out.update({k: j.get(k, out[k]) for k in ("description", "suspicious", "category",
                                                          "confidence", "reason")})
                break
            except Exception:
                continue
    if isinstance(out["suspicious"], str):
        out["suspicious"] = out["suspicious"].strip().lower() in ("true", "yes", "1")
    if out["suspicious"] is None:
        low = txt.lower()
        out["suspicious"] = bool(re.search(r"\b(suspicious|theft|steal|stealing|fight|weapon|conceal)", low)) \
            and not re.search(r"not suspicious|nothing suspicious|no suspicious", low)
    try:
        out["confidence"] = float(out["confidence"]) if out["confidence"] is not None else None
    except Exception:
        out["confidence"] = None
    return out


def parse_clock(txt: str) -> Optional[dt.datetime]:
    m = re.search(r"(20\d{2})[-/.](\d{1,2})[-/.](\d{1,2})[ T]+(\d{1,2}):(\d{2}):(\d{2})", txt)
    if m:
        try:
            return dt.datetime(*map(int, m.groups()))
        except ValueError:
            pass
    m = re.search(r"(\d{1,2})[-/.](\d{1,2})[-/.](20\d{2})[ T]+(\d{1,2}):(\d{2}):(\d{2})", txt)
    if m:
        d, mo, y, H, M, S = map(int, m.groups())
        try:
            return dt.datetime(y, mo, d, H, M, S)
        except ValueError:
            try:
                return dt.datetime(y, d, mo, H, M, S)   # month-first DVRs
            except ValueError:
                pass
    return None

In [ ]:
%%writefile /content/cctv_analysis/rules.py
"""Explainable suspicion scoring. Every flag carries a weight and a human
reason, so a reviewer can see *why* an event was ranked high."""
from __future__ import annotations

import datetime as dt
from typing import Dict, List, Optional

import numpy as np

WEIGHTS = {
    "after_hours": 30,
    "camera_tamper": 45,
    "weapon_like_object": 40,
    "restricted_zone": 25,
    "loitering": 15,
    "running": 12,
    "crowd": 10,
    "face_hidden": 18,
    "hands_up": 20,
    "crouching": 8,
    "lights_after_hours": 20,
    "semantic_match": 15,
    "vlm_suspicious": 30,
    "unknown_person_after_hours": 10,
    "repeat_visitor": 5,
    "long_event": 5,
}


def point_in_poly(x: float, y: float, poly) -> bool:
    inside = False
    n = len(poly)
    j = n - 1
    for i in range(n):
        xi, yi = poly[i]
        xj, yj = poly[j]
        if (yi > y) != (yj > y) and x < (xj - xi) * (y - yi) / (yj - yi + 1e-12) + xi:
            inside = not inside
        j = i
    return inside


def zones_for(cfg, camera: str) -> Dict[str, list]:
    z = dict(cfg.zones.get("*", {}))
    z.update(cfg.zones.get(camera, {}))
    return z


def _hm(s: str) -> dt.time:
    h, m = s.split(":")
    return dt.time(int(h), int(m))


def is_after_hours(when: Optional[dt.datetime], cfg) -> bool:
    if when is None:
        return False
    if when.weekday() in (cfg.closed_weekdays or []):
        return True
    o, c = _hm(cfg.open_hour), _hm(cfg.close_hour)
    t = when.time()
    if o <= c:
        return not (o <= t < c)
    return c <= t < o           # shop open across midnight


def score_event(ev: dict, tracks: List[dict], cfg, vlm: Optional[dict] = None,
                semantic: Optional[List[dict]] = None, tamper: Optional[List[dict]] = None) -> tuple:
    flags = []

    def add(key, reason, scale=1.0):
        flags.append({"flag": key, "points": round(WEIGHTS[key] * scale, 1), "reason": reason})

    persons = [t for t in tracks if t["cls"] == "person" and t["duration"] >= 0.5]
    start_dt = ev.get("start_dt")
    if is_after_hours(start_dt, cfg) and (persons or ev.get("motion_peak", 0) > 0.01):
        add("after_hours", f"activity at {start_dt:%H:%M} outside shop hours {cfg.open_hour}-{cfg.close_hour}"
            + (" with people" if persons else " (no person detected)"), 1.0 if persons else 0.5)
    for tp in tamper or []:
        if tp["start"] <= ev["end"] and tp["end"] >= ev["start"]:
            add("camera_tamper", f"{tp['kind']} for {tp['end'] - tp['start']:.0f}s")
            break
    weapons = sorted({t["cls"] for t in tracks if t["cls"] in cfg.weapon_classes and t["max_conf"] >= 0.4})
    if weapons:
        add("weapon_like_object", "detected: " + ", ".join(weapons))
    zone_hits = {}
    for t in persons:
        for z, secs in t.get("zones", {}).items():
            zone_hits[z] = zone_hits.get(z, 0) + secs
    zone_hits = {z: s for z, s in zone_hits.items() if s >= 1.0}
    if zone_hits:
        add("restricted_zone", "; ".join(f"{z}: {s:.0f}s" for z, s in zone_hits.items()))
    loiter = [t for t in persons if t["duration"] >= cfg.loiter_seconds]
    if loiter:
        add("loitering", f"{len(loiter)} person(s) stayed ≥{cfg.loiter_seconds:.0f}s "
                         f"(max {max(t['duration'] for t in loiter):.0f}s)")
    runners = [t for t in persons if t.get("max_speed", 0) >= cfg.running_speed]
    if runners:
        add("running", f"{len(runners)} fast-moving person(s), {max(t['max_speed'] for t in runners):.1f} body-heights/s")
    if ev.get("max_persons", 0) >= cfg.crowd_size:
        add("crowd", f"{ev['max_persons']} people at once")
    hidden = [t for t in persons if t.get("hidden_face_s", 0) >= cfg.hidden_face_seconds]
    if hidden:
        add("face_hidden", f"{len(hidden)} person(s) facing camera but no face detected "
                           f"(possible mask/helmet/cover) ≥{cfg.hidden_face_seconds:.0f}s")
    if any(t.get("hands_up_s", 0) >= 1.5 for t in persons):
        add("hands_up", "both hands raised above shoulders (possible threat / hold-up)")
    if any(t.get("crouch_s", 0) >= 3 for t in persons):
        add("crouching", "person crouching/bending low for a while")
    if ev.get("lighting_changes") and is_after_hours(start_dt, cfg):
        add("lights_after_hours", f"{ev['lighting_changes']} sudden lighting change(s) after hours (lights switched on/off?)")
    if semantic:
        best = max(semantic, key=lambda s: s["suspicious_score"])
        if best["suspicious_score"] >= cfg.semantic_alert_score and best["lift"] >= 0.6:
            add("semantic_match", f"scene resembles: '{best['top_suspicious']}' ({best['suspicious_score']:.2f})")
    if vlm and vlm.get("suspicious"):
        c = vlm.get("confidence")
        c = 0.7 if c is None else max(0.3, min(1.0, float(c)))
        add("vlm_suspicious", f"AI description: {vlm.get('category') or ''} – {vlm.get('reason') or vlm.get('description', '')[:120]}", c)
    if is_after_hours(start_dt, cfg) and any(t.get("identity") and not t.get("known_name") for t in persons):
        add("unknown_person_after_hours", "unrecognised face after hours")
    if any(t.get("identity_visits", 0) >= 5 and not t.get("known_name") for t in persons):  # distinct events
        add("repeat_visitor", "same unknown face seen in ≥5 events")
    if ev["end"] - ev["start"] >= 300 and persons:
        add("long_event", f"event lasted {(ev['end'] - ev['start']) / 60:.0f} min")
    staff_only = persons and all(t.get("known_name") for t in persons if t.get("identity"))
    score = sum(f["points"] for f in flags)
    if staff_only and any(t.get("identity") for t in persons) and not weapons:
        score *= 0.5
        flags.append({"flag": "known_people_only", "points": 0, "reason": "only known/staff faces – score halved"})
    return float(min(100.0, score)), flags


def summarize_track(t: dict, fps: float, frame_h: int) -> dict:
    """Turn raw per-frame observations into track statistics."""
    obs = t["obs"]
    times = np.array([o[0] for o in obs])
    out = {k: t[k] for k in ("track", "cls")}
    out["first"] = float(times.min())
    out["last"] = float(times.max())
    out["duration"] = float(out["last"] - out["first"])
    out["frames"] = len(obs)
    out["max_conf"] = float(max(o[1] for o in obs))
    out["mean_conf"] = float(np.mean([o[1] for o in obs]))
    cx = np.array([(o[2][0] + o[2][2]) / 2 for o in obs])
    cy = np.array([(o[2][1] + o[2][3]) / 2 for o in obs])
    hh = np.array([max(1.0, o[2][3] - o[2][1]) for o in obs])
    sp = 0.0
    if len(obs) >= 3:
        dt_ = np.diff(times)
        d = np.hypot(np.diff(cx), np.diff(cy)) / ((hh[1:] + hh[:-1]) / 2)
        v = d / np.maximum(dt_, 1e-3)
        v = np.convolve(v, np.ones(3) / 3, mode="valid") if len(v) >= 3 else v
        sp = float(np.max(v)) if len(v) else 0.0
    out["max_speed"] = sp
    out["path"] = [[round(float(a), 1), round(float(b), 1)] for a, b in zip(cx[::max(1, len(cx) // 40)], cy[::max(1, len(cy) // 40)])]
    step = 1.0 / fps
    out["zones"] = {z: n * step for z, n in t.get("zone_frames", {}).items()}
    out["hidden_face_s"] = t.get("hidden_frames", 0) * step
    out["hands_up_s"] = t.get("hands_up", 0) * step
    out["crouch_s"] = t.get("crouch", 0) * step
    out["faces_found"] = t.get("faces_found", 0)
    return out

In [ ]:
%%writefile /content/cctv_analysis/annotate.py
"""Drawing helpers: boxes, labels, header bars, zones, contact sheets, heatmaps."""
from __future__ import annotations

from typing import List, Optional

import cv2
import numpy as np

PALETTE = [(56, 56, 255), (151, 157, 255), (31, 112, 255), (29, 178, 255), (49, 210, 207),
           (10, 249, 72), (23, 204, 146), (134, 219, 61), (52, 147, 26), (187, 212, 0),
           (168, 153, 44), (255, 194, 0), (147, 69, 52), (255, 115, 100), (236, 24, 0),
           (255, 56, 132), (133, 0, 82), (255, 56, 203), (200, 149, 255), (199, 55, 255)]


def color_for(i) -> tuple:
    if i is None:
        return PALETTE[0]
    k = i if isinstance(i, int) else sum(map(ord, str(i)))
    return PALETTE[(k * 7) % len(PALETTE)]


def label(img, text, x, y, color, scale=0.5):
    (tw, th), bl = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 1)
    y = max(th + 4, y)
    cv2.rectangle(img, (x, y - th - 4), (x + tw + 4, y + bl - 2), color, -1)
    cv2.putText(img, text, (x + 2, y - 3), cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), 1, cv2.LINE_AA)


def draw_zones(img, zones: dict):
    h, w = img.shape[:2]
    over = img.copy()
    for name, poly in zones.items():
        pts = np.array([[int(x * w), int(y * h)] for x, y in poly], np.int32)
        cv2.fillPoly(over, [pts], (0, 0, 255))
        cv2.polylines(img, [pts], True, (0, 0, 255), 2)
        label(img, name, int(pts[:, 0].min()), int(pts[:, 1].min()) + 14, (0, 0, 180))
    cv2.addWeighted(over, 0.15, img, 0.85, 0, img)


def draw_dets(frame, dets: List[dict], faces: List[dict] = (), ids: Optional[dict] = None,
              header: str = "", zones: Optional[dict] = None) -> np.ndarray:
    img = frame.copy()
    s = max(0.45, img.shape[1] / 1600)
    lw = max(1, int(round(img.shape[1] / 640)))
    if zones:
        draw_zones(img, zones)
    for d in dets:
        x1, y1, x2, y2 = [int(v) for v in d["xyxy"]]
        c = color_for(d.get("track"))
        cv2.rectangle(img, (x1, y1), (x2, y2), c, lw + 1)
        txt = f"{d['cls']} {d['conf']:.2f}"
        if d.get("track") is not None:
            txt = f"#{d['track']} " + txt
            if ids and ids.get(d["track"]):
                txt += f" [{ids[d['track']]}]"
        label(img, txt, x1, y1, c, s)
    for f in faces:
        x1, y1, x2, y2 = [int(v) for v in f["box"]]
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 255), lw)
        label(img, f"face {f['score']:.2f}", x1, y2 + 16, (0, 140, 140), s * 0.9)
    if header:
        bar = int(28 * s / 0.5)
        cv2.rectangle(img, (0, img.shape[0] - bar), (img.shape[1], img.shape[0]), (0, 0, 0), -1)
        cv2.putText(img, header, (8, img.shape[0] - int(bar * 0.3)), cv2.FONT_HERSHEY_SIMPLEX,
                    s * 1.1, (255, 255, 255), 1, cv2.LINE_AA)
    return img


def contact_sheet(frames: List[np.ndarray], captions: List[str], cols: int = 3, tile_w: int = 480) -> np.ndarray:
    if not frames:
        return np.zeros((10, 10, 3), np.uint8)
    tiles = []
    for f, c in zip(frames, captions):
        h = int(f.shape[0] * tile_w / f.shape[1])
        t = cv2.resize(f, (tile_w, h), interpolation=cv2.INTER_AREA)
        label(t, c, 4, 18, (40, 40, 40), 0.5)
        tiles.append(t)
    th = max(t.shape[0] for t in tiles)
    tiles = [cv2.copyMakeBorder(t, 0, th - t.shape[0], 0, 0, cv2.BORDER_CONSTANT) for t in tiles]
    cols = min(cols, len(tiles))
    while len(tiles) % cols:
        tiles.append(np.zeros_like(tiles[0]))
    rows = [np.hstack(tiles[i:i + cols]) for i in range(0, len(tiles), cols)]
    return np.vstack(rows)


def heatmap_overlay(base_gray_or_bgr: np.ndarray, heat: np.ndarray) -> np.ndarray:
    base = base_gray_or_bgr
    if base.ndim == 2:
        base = cv2.cvtColor(base, cv2.COLOR_GRAY2BGR)
    hm = cv2.resize(heat, (base.shape[1], base.shape[0]))
    hm = np.uint8(255 * np.sqrt(np.clip(hm, 0, 1)))
    col = cv2.applyColorMap(hm, cv2.COLORMAP_JET)
    return cv2.addWeighted(base, 0.55, col, 0.45, 0)


def thumb(img: np.ndarray, w: int = 320) -> np.ndarray:
    h = int(img.shape[0] * w / img.shape[1])
    return cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA)

In [ ]:
%%writefile /content/cctv_analysis/storage.py
"""SQLite store: the single source of truth. Makes runs resumable (finished
videos are skipped) and cumulative (reports always cover everything)."""
from __future__ import annotations

import json
import os
import shutil
import sqlite3
from typing import Any, Dict, Iterable, List

SCHEMA = """
CREATE TABLE IF NOT EXISTS videos (
  id INTEGER PRIMARY KEY AUTOINCREMENT, path TEXT UNIQUE, rel_path TEXT, fingerprint TEXT,
  camera TEXT, start_dt TEXT, end_dt TEXT, time_source TEXT, duration REAL, fps REAL, width INT,
  height INT, codec TEXT, size_mb REAL, status TEXT, error TEXT, n_events INT, motion_seconds REAL,
  processed_at TEXT, proc_seconds REAL, heatmap_path TEXT, drive_link TEXT, models TEXT,
  lighting_changes INT, osd_masked INT);
CREATE TABLE IF NOT EXISTS events (
  event_id TEXT PRIMARY KEY, video_id INT, video_path TEXT, rel_path TEXT, camera TEXT,
  start_off REAL, end_off REAL, start_dt TEXT, end_dt TEXT, duration REAL, kind TEXT,
  motion_peak REAL, motion_mean REAL, max_persons INT, n_tracks INT, n_person_tracks INT,
  objects TEXT, faces_detected INT, identities TEXT, best_conf REAL, score REAL, level TEXT,
  flags TEXT, semantic TEXT, vlm TEXT, snapshots TEXT, contact_sheet TEXT, clip_path TEXT,
  folder TEXT, after_hours INT, frames_analyzed INT);
CREATE TABLE IF NOT EXISTS tracks (
  event_id TEXT, track INT, cls TEXT, first_off REAL, last_off REAL, first_dt TEXT, duration REAL,
  frames INT, max_conf REAL, mean_conf REAL, max_speed REAL, zones TEXT, identity TEXT,
  identity_sim REAL, known_name TEXT, face_crop TEXT, person_crop TEXT, hidden_face_s REAL,
  posture TEXT, faces_found INT, path TEXT, PRIMARY KEY (event_id, track));
CREATE TABLE IF NOT EXISTS faces (
  id INTEGER PRIMARY KEY AUTOINCREMENT, event_id TEXT, track INT, t_off REAL, dt TEXT, score REAL,
  quality REAL, box TEXT, crop_path TEXT, identity TEXT, similarity REAL, known_name TEXT);
CREATE TABLE IF NOT EXISTS detections (
  event_id TEXT, t_off REAL, dt TEXT, cls TEXT, conf REAL, track INT,
  x1 REAL, y1 REAL, x2 REAL, y2 REAL);
CREATE TABLE IF NOT EXISTS tamper (
  video_id INT, start_off REAL, end_off REAL, start_dt TEXT, kind TEXT);
CREATE INDEX IF NOT EXISTS ix_ev_video ON events(video_id);
CREATE INDEX IF NOT EXISTS ix_det_ev ON detections(event_id);
"""


class Store:
    def __init__(self, local_path: str, remote_path: str = ""):
        self.local = local_path
        self.remote = remote_path
        os.makedirs(os.path.dirname(local_path) or ".", exist_ok=True)
        if remote_path and os.path.exists(remote_path) and not os.path.exists(local_path):
            shutil.copy2(remote_path, local_path)
        self.db = sqlite3.connect(local_path)
        self.db.row_factory = sqlite3.Row
        self.db.executescript(SCHEMA)
        self.db.commit()

    # ------------------------------------------------------------------ util
    @staticmethod
    def _enc(v):
        if isinstance(v, (dict, list, tuple)):
            return json.dumps(v, default=str)
        return v

    def insert(self, table: str, row: Dict[str, Any], replace=True):
        keys = list(row)
        q = f"INSERT OR {'REPLACE' if replace else 'IGNORE'} INTO {table} ({','.join(keys)}) VALUES ({','.join('?' * len(keys))})"
        cur = self.db.execute(q, [self._enc(row[k]) for k in keys])
        return cur.lastrowid

    def insert_many(self, table: str, rows: Iterable[Dict[str, Any]]):
        rows = list(rows)
        if not rows:
            return
        keys = list(rows[0])
        q = f"INSERT INTO {table} ({','.join(keys)}) VALUES ({','.join('?' * len(keys))})"
        self.db.executemany(q, [[self._enc(r.get(k)) for k in keys] for r in rows])

    def query(self, sql: str, args=()) -> List[dict]:
        return [dict(r) for r in self.db.execute(sql, args).fetchall()]

    # --------------------------------------------------------------- videos
    def video_done(self, path: str, fingerprint: str) -> bool:
        r = self.db.execute("SELECT status, fingerprint FROM videos WHERE path=?", (path,)).fetchone()
        return bool(r and r["status"] == "done" and r["fingerprint"] == fingerprint)

    def start_video(self, row: dict) -> int:
        old = self.db.execute("SELECT id FROM videos WHERE path=?", (row["path"],)).fetchone()
        if old:
            vid = old["id"]
            self.purge_video(vid)
            sets = ",".join(f"{k}=?" for k in row)
            self.db.execute(f"UPDATE videos SET {sets} WHERE id=?", [self._enc(v) for v in row.values()] + [vid])
        else:
            vid = self.insert("videos", row)
        self.db.commit()
        return vid

    def purge_video(self, vid: int):
        evs = [r["event_id"] for r in self.query("SELECT event_id FROM events WHERE video_id=?", (vid,))]
        for t in ("tracks", "faces", "detections"):
            self.db.executemany(f"DELETE FROM {t} WHERE event_id=?", [(e,) for e in evs])
        self.db.execute("DELETE FROM events WHERE video_id=?", (vid,))
        self.db.execute("DELETE FROM tamper WHERE video_id=?", (vid,))

    def update_video(self, vid: int, **kw):
        sets = ",".join(f"{k}=?" for k in kw)
        self.db.execute(f"UPDATE videos SET {sets} WHERE id=?", [self._enc(v) for v in kw.values()] + [vid])
        self.db.commit()

    def commit(self, sync: bool = True):
        self.db.commit()
        if sync and self.remote:
            try:
                os.makedirs(os.path.dirname(self.remote), exist_ok=True)
                bk = sqlite3.connect(self.remote + ".tmp")
                self.db.backup(bk)
                bk.close()
                os.replace(self.remote + ".tmp", self.remote)
            except Exception:
                pass

    def close(self):
        self.commit()
        self.db.close()

In [ ]:
%%writefile /content/cctv_analysis/report.py
"""Reports built from the SQLite store: Excel workbook (with thumbnails,
clickable citations, review columns, charts), HTML dashboard, PNG charts and a
highlight reel of the most suspicious events."""
from __future__ import annotations

import base64
import csv
import datetime as dt
import html
import json
import logging
import os
import subprocess
from collections import Counter, defaultdict

import cv2

from . import video_io as V
from .rules import WEIGHTS

log = logging.getLogger("cctv")
LEVEL_COLORS = {"HIGH": "#F8696B", "REVIEW": "#FFB347", "LOW": "#FFF2A8", "NORMAL": "#E2EFDA"}
EXCEL_MAX_ROWS = 1_000_000


def _j(s, default=None):
    if s in (None, ""):
        return default
    try:
        return json.loads(s)
    except Exception:
        return default


def _rel(path: str, base: str) -> str:
    if not path:
        return ""
    try:
        return os.path.relpath(path, base)
    except ValueError:
        return path


def _thumb(src: str, out_dir: str, w: int = 200) -> str:
    if not src or not os.path.exists(src):
        return ""
    dst = os.path.join(out_dir, "thumbs", os.path.relpath(src, out_dir).replace(os.sep, "__"))
    dst = os.path.splitext(dst)[0] + ".jpg"
    if os.path.exists(dst):
        return dst
    img = cv2.imread(src)
    if img is None:
        return ""
    h = max(1, int(img.shape[0] * w / img.shape[1]))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    cv2.imwrite(dst, cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA), [cv2.IMWRITE_JPEG_QUALITY, 80])
    return dst


def _flags_text(flags) -> str:
    return "; ".join(f"{f['flag']} (+{f['points']:g}): {f['reason']}" for f in flags or [])


def _vlm_text(v) -> str:
    if not v:
        return ""
    d = v.get("description") or ""
    if v.get("suspicious"):
        d = f"[SUSPICIOUS: {v.get('category') or 'yes'}] " + d + (f" — {v['reason']}" if v.get("reason") else "")
    return d


def load(store):
    events = store.query("SELECT * FROM events ORDER BY COALESCE(start_dt, ''), camera, start_off")
    for e in events:
        for k in ("flags", "semantic", "vlm", "snapshots", "objects", "identities"):
            e[k] = _j(e[k], [] if k != "vlm" and k != "objects" else ({} if k == "objects" else None))
    videos = store.query("SELECT * FROM videos ORDER BY camera, start_dt, path")
    tracks = store.query("SELECT t.*, e.camera, e.video_path, e.rel_path, e.start_dt AS ev_dt FROM tracks t "
                         "JOIN events e USING(event_id) ORDER BY t.first_dt, t.event_id, t.track")
    faces = store.query("SELECT f.*, e.camera, e.rel_path, e.video_path FROM faces f JOIN events e USING(event_id) "
                        "ORDER BY f.dt, f.event_id")
    tamper = store.query("SELECT t.*, v.rel_path, v.camera FROM tamper t JOIN videos v ON v.id=t.video_id "
                         "ORDER BY t.start_dt")
    return events, videos, tracks, faces, tamper


def identities_table(tracks):
    ids = defaultdict(lambda: {"tracks": 0, "events": set(), "cameras": set(), "first": None, "last": None,
                               "name": "", "sample": "", "best_sim": 0})
    for t in tracks:
        if not t["identity"]:
            continue
        r = ids[t["identity"]]
        r["tracks"] += 1
        r["events"].add(t["event_id"])
        r["cameras"].add(t["camera"])
        when = t["first_dt"] or ""
        r["first"] = min(filter(None, [r["first"], when]), default=when)
        r["last"] = max(filter(None, [r["last"], when]), default=when)
        if t["known_name"]:
            r["name"] = t["known_name"]
        if t["face_crop"] and (t["identity_sim"] or 0) >= r["best_sim"]:
            r["sample"], r["best_sim"] = t["face_crop"], t["identity_sim"] or 0
    return dict(sorted(ids.items(), key=lambda kv: -len(kv[1]["events"])))


# ====================================================================== Excel
def build_excel(store, cfg, path):
    import xlsxwriter
    out = cfg.output_dir
    events, videos, tracks, faces, tamper = load(store)
    idt = identities_table(tracks)
    wb = xlsxwriter.Workbook(path, {"strings_to_urls": False, "nan_inf_to_errors": True})
    F = {
        "h": wb.add_format({"bold": True, "bg_color": "#1F3864", "font_color": "white", "text_wrap": True,
                            "valign": "vcenter", "border": 1}),
        "title": wb.add_format({"bold": True, "font_size": 16, "font_color": "#1F3864"}),
        "b": wb.add_format({"bold": True}),
        "wrap": wb.add_format({"text_wrap": True, "valign": "top"}),
        "top": wb.add_format({"valign": "top"}),
        "link": wb.add_format({"font_color": "blue", "underline": 1, "valign": "top"}),
        "num": wb.add_format({"num_format": "0.00", "valign": "top"}),
        "int": wb.add_format({"num_format": "0", "valign": "top"}),
        "pct": wb.add_format({"num_format": "0.0%", "valign": "top"}),
    }
    for lv, c in LEVEL_COLORS.items():
        F[lv] = wb.add_format({"bg_color": c, "bold": True, "valign": "top", "align": "center"})

    def link(ws, r, c, target, text):
        if target:
            ws.write_url(r, c, target if target.startswith("http") else "external:" + target, F["link"], text)
        else:
            ws.write_blank(r, c, None)

    thumbs_used = [0]

    def image(ws, r, c, src, w=150):
        if not cfg.excel_thumbnails or thumbs_used[0] >= cfg.excel_max_thumbnails:
            return
        t = _thumb(src, out, 200)
        if t:
            img = cv2.imread(t)
            if img is None:
                return
            sc = w / img.shape[1]
            ws.insert_image(r, c, t, {"x_scale": sc, "y_scale": sc, "x_offset": 2, "y_offset": 2,
                                      "object_position": 1})
            thumbs_used[0] += 1

    # ---------------------------------------------------------- Read Me
    ws = wb.add_worksheet("Read Me")
    ws.set_column(0, 0, 28)
    ws.set_column(1, 1, 110, F["wrap"])
    ws.write(0, 0, "CCTV Analysis Report", F["title"])
    ws.write(1, 0, "Generated")
    ws.write(1, 1, dt.datetime.now().strftime("%Y-%m-%d %H:%M"))
    rows = [
        ("How to review", "Start with the 'Events' sheet. It is sorted by time; filter the Level column to HIGH/REVIEW "
                          "or sort by Score. Each row cites the exact video file, the offset inside the video "
                          "(HH:MM:SS) and the real clock time. Click 'Clip' to watch the event, 'Snapshot' or "
                          "'Contact sheet' for stills, 'Video (Drive)' to open the original recording."),
        ("Links", "File links are relative to this workbook. They work when the workbook is opened from the "
                  "output folder (Google Drive for desktop, or after downloading the whole folder). "
                  "Drive links open in the browser."),
        ("Seek in VLC", "Open the original file in VLC and jump to 'Video offset start' (Ctrl+T)."),
        ("Score / Level", "Score 0-100 is the sum of explainable flags (see 'Flags'). HIGH ≥ 70, REVIEW ≥ "
                          f"{cfg.suspicion_alert_threshold:g}, LOW > 0, NORMAL = 0. It is a triage aid, not proof."),
        ("Identities", "P#### are anonymous face clusters that persist between runs (same person ≈ same ID). "
                       "Put photos in the known-faces folder (one sub-folder per person) to label staff/owner."),
        ("Review columns", "'Review status' and 'Notes' are for you; they are kept only in this file."),
        ("Shop hours", f"{cfg.open_hour}–{cfg.close_hour}, timezone {cfg.timezone}. Activity outside = after_hours."),
    ]
    for i, (a, b) in enumerate(rows, 3):
        ws.write(i, 0, a, F["b"])
        ws.write(i, 1, b)
    r0 = len(rows) + 4
    ws.write(r0, 0, "Flag", F["h"])
    ws.write(r0, 1, "Points (max) / meaning", F["h"])
    meaning = {
        "after_hours": "movement/people outside opening hours", "camera_tamper": "lens covered, blackout, blur or camera moved",
        "weapon_like_object": "knife / scissors / bat detected", "restricted_zone": "person inside a configured zone (e.g. cash counter)",
        "loitering": f"person stayed ≥ {cfg.loiter_seconds:g}s", "running": "very fast movement",
        "crowd": f"≥ {cfg.crowd_size} people at once", "face_hidden": "person facing camera but face not visible (mask/helmet)",
        "hands_up": "both hands above shoulders", "crouching": "crouching / bending low",
        "lights_after_hours": "lights switched on/off while the shop is closed", "semantic_match": "SigLIP-2 image-text match with a suspicious description",
        "vlm_suspicious": "the vision-language model judged the event suspicious",
        "unknown_person_after_hours": "unrecognised face after hours", "repeat_visitor": "same unknown face in ≥ 5 events",
        "long_event": "event ≥ 5 min with people"}
    for i, (k, v) in enumerate(WEIGHTS.items(), r0 + 1):
        ws.write(i, 0, k)
        ws.write(i, 1, f"{v} — {meaning.get(k, '')}")

    # ---------------------------------------------------------- Summary
    ws = wb.add_worksheet("Summary")
    ws.set_column(0, 0, 34)
    ws.set_column(1, 8, 16)
    ws.write(0, 0, "Summary", F["title"])
    tot_dur = sum(v["duration"] or 0 for v in videos)
    tot_motion = sum(v["motion_seconds"] or 0 for v in videos)
    lv = Counter(e["level"] for e in events)
    kpis = [
        ("Videos analysed", sum(1 for v in videos if v["status"] == "done")),
        ("Videos failed", sum(1 for v in videos if v["status"] not in ("done",))),
        ("Footage analysed (hours)", round(tot_dur / 3600, 2)),
        ("Motion time (hours)", round(tot_motion / 3600, 2)),
        ("Footage you can skip", f"{(1 - tot_motion / tot_dur) * 100:.1f}%" if tot_dur else "-"),
        ("Events", len(events)),
        ("Events with people", sum(1 for e in events if e["n_person_tracks"])),
        ("HIGH", lv.get("HIGH", 0)), ("REVIEW", lv.get("REVIEW", 0)), ("LOW", lv.get("LOW", 0)),
        ("NORMAL", lv.get("NORMAL", 0)),
        ("Person tracks", sum(1 for t in tracks if t["cls"] == "person")),
        ("Face samples", len(faces)),
        ("Distinct face identities", len(idt)),
        ("Tamper incidents", len(tamper)),
        ("Date range", f"{min((e['start_dt'] for e in events if e['start_dt']), default='-')} → "
                       f"{max((e['end_dt'] for e in events if e['end_dt']), default='-')}"),
    ]
    for i, (a, b) in enumerate(kpis, 2):
        ws.write(i, 0, a, F["b"])
        ws.write(i, 1, b)
    r = len(kpis) + 3
    ws.write(r, 0, "Top suspicious events", F["b"])
    hdr = ["Event ID", "Level", "Score", "Camera", "Start", "Video", "Offset", "Main reasons"]
    for c, h in enumerate(hdr):
        ws.write(r + 1, c, h, F["h"])
    top = sorted(events, key=lambda e: -(e["score"] or 0))[:25]
    for i, e in enumerate(top, r + 2):
        ws.write_url(i, 0, f"internal:'Events'!A{events.index(e) + 2}", F["link"], e["event_id"])
        ws.write(i, 1, e["level"], F.get(e["level"]))
        ws.write(i, 2, e["score"])
        ws.write(i, 3, e["camera"])
        ws.write(i, 4, e["start_dt"] or "")
        ws.write(i, 5, e["rel_path"])
        ws.write(i, 6, V.fmt_offset(e["start_off"]))
        ws.write(i, 7, ", ".join(f["flag"] for f in e["flags"]))
    ws.set_column(7, 7, 60)
    # hourly table + native chart
    hr = r + 3 + len(top) + 1
    ws.write(hr, 0, "Events by hour of day", F["b"])
    ws.write(hr + 1, 0, "Hour", F["h"])
    ws.write(hr + 1, 1, "Events", F["h"])
    ws.write(hr + 1, 2, "With people", F["h"])
    ws.write(hr + 1, 3, "REVIEW+HIGH", F["h"])
    byh = defaultdict(lambda: [0, 0, 0])
    for e in events:
        if e["start_dt"]:
            h = int(e["start_dt"][11:13])
            byh[h][0] += 1
            byh[h][1] += bool(e["n_person_tracks"])
            byh[h][2] += e["level"] in ("HIGH", "REVIEW")
    for h in range(24):
        ws.write(hr + 2 + h, 0, f"{h:02d}:00")
        for c in range(3):
            ws.write(hr + 2 + h, 1 + c, byh[h][c])
    ch = wb.add_chart({"type": "column"})
    for c, name in ((1, "Events"), (3, "REVIEW+HIGH")):
        ch.add_series({"name": name, "categories": ["Summary", hr + 2, 0, hr + 25, 0],
                       "values": ["Summary", hr + 2, c, hr + 25, c]})
    ch.set_title({"name": "Activity by hour"})
    ch.set_size({"width": 720, "height": 300})
    ws.insert_chart(2, 3, ch)
    # per day per camera
    byd = defaultdict(Counter)
    for e in events:
        byd[(e["start_dt"] or "unknown")[:10]][e["camera"]] += 1
    cams = sorted({e["camera"] for e in events})
    dr = hr + 28
    ws.write(dr, 0, "Events per day and camera", F["b"])
    ws.write(dr + 1, 0, "Date", F["h"])
    for c, cam in enumerate(cams, 1):
        ws.write(dr + 1, c, cam, F["h"])
    for i, (d, cnt) in enumerate(sorted(byd.items()), dr + 2):
        ws.write(i, 0, d)
        for c, cam in enumerate(cams, 1):
            ws.write(i, c, cnt.get(cam, 0))

    # ---------------------------------------------------------- Events
    ws = wb.add_worksheet("Events")
    cols = [("Event ID", 26), ("Level", 9), ("Score", 7), ("Camera", 10), ("Date", 11), ("Start time", 10),
            ("End time", 10), ("Duration (s)", 9), ("Video file", 34), ("Video offset start", 10),
            ("Video offset end", 10), ("Seek (s)", 8), ("Kind", 11), ("Max people at once", 8),
            ("Person tracks", 8), ("Objects (unique tracks)", 26), ("Faces detected", 8), ("Identities", 18),
            ("Best detection conf", 9), ("Flags / reasons", 60), ("AI description (VLM)", 60),
            ("Scene tag (SigLIP-2)", 34), ("Snapshot", 24), ("Snapshot file", 12), ("Contact sheet", 12),
            ("Clip", 8), ("Video (Drive)", 10), ("Event folder", 12), ("After hours", 7), ("Motion peak", 8),
            ("Frames analysed", 8), ("Time source", 14), ("Review status", 14), ("Notes", 30)]
    for c, (h, w) in enumerate(cols):
        ws.write(0, c, h, F["h"])
        ws.set_column(c, c, w)
    vmap = {v["id"]: v for v in videos}
    snap_col = [c for c, (h, _) in enumerate(cols) if h == "Snapshot"][0]
    for i, e in enumerate(events, 1):
        v = vmap.get(e["video_id"], {})
        snaps = e["snapshots"] or []
        best = max(snaps, key=lambda s: s.get("persons", 0)) if snaps else {}
        sem = max(e["semantic"] or [{}], key=lambda s: s.get("suspicious_score", 0)) if e["semantic"] else {}
        row = [e["event_id"], e["level"], e["score"], e["camera"], (e["start_dt"] or "")[:10],
               (e["start_dt"] or "")[11:], (e["end_dt"] or "")[11:], e["duration"], e["rel_path"],
               V.fmt_offset(e["start_off"]), V.fmt_offset(e["end_off"]), round(e["start_off"], 1), e["kind"],
               e["max_persons"], e["n_person_tracks"],
               ", ".join(f"{k}×{n}" for k, n in sorted((e["objects"] or {}).items(), key=lambda kv: -kv[1])),
               e["faces_detected"], ", ".join(e["identities"] or []), e["best_conf"], _flags_text(e["flags"]),
               _vlm_text(e["vlm"]),
               f"{sem.get('top_label', '')} ({sem.get('top_score', 0):.2f})" if sem else ""]
        for c, val in enumerate(row):
            if c == 1:
                ws.write(i, c, val, F.get(val, F["top"]))
            elif c in (19, 20):
                ws.write(i, c, val, F["wrap"])
            else:
                ws.write(i, c, val, F["top"])
        c = snap_col
        image(ws, i, c, best.get("annotated", ""), 160)
        link(ws, i, c + 1, _rel(best.get("annotated", ""), out), "open")
        link(ws, i, c + 2, _rel(e["contact_sheet"], out), "sheet")
        link(ws, i, c + 3, _rel(e["clip_path"], out), "clip" if e["clip_path"] else "")
        link(ws, i, c + 4, v.get("drive_link") or "", "drive" if v.get("drive_link") else "")
        link(ws, i, c + 5, _rel(e["folder"], out), "folder")
        ws.write(i, c + 6, "yes" if e["after_hours"] else "")
        ws.write(i, c + 7, e["motion_peak"])
        ws.write(i, c + 8, e["frames_analyzed"])
        ws.write(i, c + 9, v.get("time_source", ""))
        ws.set_row(i, 92)
    n = len(events)
    if n:
        ws.autofilter(0, 0, n, len(cols) - 1)
        ws.conditional_format(1, 2, n, 2, {"type": "3_color_scale", "min_color": "#63BE7B",
                                           "mid_color": "#FFEB84", "max_color": "#F8696B"})
        ws.data_validation(1, len(cols) - 2, n, len(cols) - 2, {
            "validate": "list", "source": ["Not reviewed", "OK / normal", "Suspicious", "Theft confirmed",
                                           "Reported", "False alarm"]})
    ws.freeze_panes(1, 1)

    # ---------------------------------------------------------- People / tracks
    ws = wb.add_worksheet("People & Objects")
    cols = [("Event ID", 26), ("Track", 6), ("Class", 10), ("Camera", 9), ("First seen", 19), ("Seen for (s)", 8),
            ("Video file", 30), ("Video offset", 10), ("Max conf", 7), ("Mean conf", 7), ("Max speed (body-h/s)", 9),
            ("Zones (s)", 18), ("Identity", 9), ("Known name", 12), ("Match similarity", 8), ("Faces found", 7),
            ("Face hidden (s)", 8), ("Posture (s)", 26), ("Person crop", 14), ("Face", 12), ("Crop file", 8),
            ("Face file", 8)]
    for c, (h, w) in enumerate(cols):
        ws.write(0, c, h, F["h"])
        ws.set_column(c, c, w)
    for i, t in enumerate(tracks[:EXCEL_MAX_ROWS - 1], 1):
        z = _j(t["zones"], {}) or {}
        p = _j(t["posture"], {}) or {}
        vals = [t["event_id"], t["track"], t["cls"], t["camera"], t["first_dt"] or "", t["duration"], t["rel_path"],
                V.fmt_offset(t["first_off"]), t["max_conf"], t["mean_conf"], t["max_speed"],
                ", ".join(f"{k}: {v:g}" for k, v in z.items()), t["identity"] or "", t["known_name"] or "",
                t["identity_sim"], t["faces_found"], t["hidden_face_s"],
                ", ".join(f"{k}={v:g}" for k, v in p.items() if v)]
        for c, val in enumerate(vals):
            ws.write(i, c, val, F["top"])
        big = i <= cfg.excel_max_thumbnails and (t["person_crop"] or t["face_crop"])
        if big:
            image(ws, i, 18, t["person_crop"], 60)
            image(ws, i, 19, t["face_crop"], 60)
            ws.set_row(i, 90)
        link(ws, i, 20, _rel(t["person_crop"], out), "crop" if t["person_crop"] else "")
        link(ws, i, 21, _rel(t["face_crop"], out), "face" if t["face_crop"] else "")
    if tracks:
        ws.autofilter(0, 0, min(len(tracks), EXCEL_MAX_ROWS - 1), len(cols) - 1)
    ws.freeze_panes(1, 1)

    # ---------------------------------------------------------- Identities
    ws = wb.add_worksheet("Face Identities")
    cols = [("Identity", 10), ("Known name", 14), ("Sample face", 14), ("Events", 8), ("Appearances", 8),
            ("Cameras", 14), ("First seen", 19), ("Last seen", 19), ("Event IDs", 80)]
    for c, (h, w) in enumerate(cols):
        ws.write(0, c, h, F["h"])
        ws.set_column(c, c, w)
    for i, (pid, r_) in enumerate(idt.items(), 1):
        vals = [pid, r_["name"], "", len(r_["events"]), r_["tracks"], ", ".join(sorted(r_["cameras"])),
                r_["first"], r_["last"], ", ".join(sorted(r_["events"]))]
        for c, val in enumerate(vals):
            ws.write(i, c, val, F["wrap"] if c == 8 else F["top"])
        image(ws, i, 2, r_["sample"], 80)
        ws.set_row(i, 85)
    ws.freeze_panes(1, 1)

    # ---------------------------------------------------------- Faces
    ws = wb.add_worksheet("Faces")
    cols = [("Event ID", 26), ("Track", 6), ("Time", 19), ("Camera", 9), ("Video file", 30), ("Video offset", 10),
            ("Detector score", 8), ("Quality", 7), ("Identity", 9), ("Known name", 12), ("Similarity", 8),
            ("Face", 12), ("Face file", 8)]
    for c, (h, w) in enumerate(cols):
        ws.write(0, c, h, F["h"])
        ws.set_column(c, c, w)
    for i, f in enumerate(faces[:EXCEL_MAX_ROWS - 1], 1):
        vals = [f["event_id"], f["track"], f["dt"] or "", f["camera"], f["rel_path"], V.fmt_offset(f["t_off"]),
                f["score"], f["quality"], f["identity"] or "", f["known_name"] or "", f["similarity"]]
        for c, val in enumerate(vals):
            ws.write(i, c, val, F["top"])
        if i <= 1500:
            image(ws, i, 11, f["crop_path"], 60)
            ws.set_row(i, 70)
        link(ws, i, 12, _rel(f["crop_path"], out), "face")
    if faces:
        ws.autofilter(0, 0, min(len(faces), EXCEL_MAX_ROWS - 1), len(cols) - 1)
    ws.freeze_panes(1, 1)

    # ---------------------------------------------------------- Detections
    ws = wb.add_worksheet("Detections (1 per s)")
    cols = ["Event ID", "Video offset (s)", "Time", "Class", "Confidence", "Track", "x1", "y1", "x2", "y2"]
    for c, h in enumerate(cols):
        ws.write(0, c, h, F["h"])
    ws.set_column(0, 0, 26)
    ws.set_column(2, 2, 19)
    cur = store.db.execute("SELECT event_id,t_off,dt,cls,conf,track,x1,y1,x2,y2 FROM detections ORDER BY dt, event_id, t_off")
    csv_path = os.path.join(out, "detections_full.csv")
    i = 0
    with open(csv_path, "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(cols)
        for row in cur:
            row = list(row)
            w.writerow(row)
            i += 1
            if i < EXCEL_MAX_ROWS:
                ws.write_row(i, 0, row)
    if i:
        ws.autofilter(0, 0, min(i, EXCEL_MAX_ROWS - 1), len(cols) - 1)
    ws.freeze_panes(1, 0)

    # ---------------------------------------------------------- Videos
    ws = wb.add_worksheet("Videos")
    cols = [("Video file", 40), ("Camera", 9), ("Recording start", 19), ("Recording end", 19), ("Time source", 16),
            ("Duration", 10), ("Resolution", 11), ("FPS", 6), ("Codec", 8), ("Size (MB)", 9), ("Events", 7),
            ("Motion (s)", 9), ("Motion %", 8), ("Lighting changes", 8), ("Clock masked", 7),
            ("Processing (s)", 9), ("Speed (x real-time)", 9), ("Status", 8), ("Error", 40), ("Heatmap", 10),
            ("Drive link", 10), ("Models", 40)]
    for c, (h, w) in enumerate(cols):
        ws.write(0, c, h, F["h"])
        ws.set_column(c, c, w)
    for i, v in enumerate(videos, 1):
        d = v["duration"] or 0
        vals = [v["rel_path"], v["camera"], v["start_dt"] or "", v["end_dt"] or "", v["time_source"],
                V.fmt_offset(d), f"{v['width']}x{v['height']}", round(v["fps"] or 0, 2), v["codec"],
                round(v["size_mb"] or 0, 1), v["n_events"], round(v["motion_seconds"] or 0, 1)]
        for c, val in enumerate(vals):
            ws.write(i, c, val, F["top"])
        ws.write(i, 12, (v["motion_seconds"] or 0) / d if d else 0, F["pct"])
        ws.write(i, 13, v["lighting_changes"])
        ws.write(i, 14, "yes" if v["osd_masked"] else "")
        ws.write(i, 15, round(v["proc_seconds"] or 0, 1))
        ws.write(i, 16, round(d / v["proc_seconds"], 1) if v["proc_seconds"] else "")
        ws.write(i, 17, v["status"])
        ws.write(i, 18, v["error"] or "", F["wrap"])
        link(ws, i, 19, _rel(v["heatmap_path"], out), "heatmap" if v["heatmap_path"] else "")
        link(ws, i, 20, v["drive_link"] or "", "drive" if v["drive_link"] else "")
        ws.write(i, 21, v["models"] or "")
    ws.freeze_panes(1, 1)

    # ---------------------------------------------------------- Tamper
    ws = wb.add_worksheet("Tamper & Camera")
    cols = [("Video file", 40), ("Camera", 9), ("Start", 19), ("Video offset", 10), ("Duration (s)", 9), ("Kind", 30)]
    for c, (h, w) in enumerate(cols):
        ws.write(0, c, h, F["h"])
        ws.set_column(c, c, w)
    for i, t in enumerate(tamper, 1):
        ws.write_row(i, 0, [t["rel_path"], t["camera"], t["start_dt"] or "", V.fmt_offset(t["start_off"]),
                            round(t["end_off"] - t["start_off"], 1), t["kind"]])

    # ---------------------------------------------------------- Hourly heat table
    ws = wb.add_worksheet("Activity Grid")
    ws.write(0, 0, "People-events per date × hour (colour = busier)", F["b"])
    ws.write(1, 0, "Date", F["h"])
    for h in range(24):
        ws.write(1, 1 + h, f"{h:02d}", F["h"])
    grid = defaultdict(Counter)
    for e in events:
        if e["start_dt"] and e["n_person_tracks"]:
            grid[e["start_dt"][:10]][int(e["start_dt"][11:13])] += 1
    for i, d in enumerate(sorted(grid), 2):
        ws.write(i, 0, d)
        for h in range(24):
            ws.write(i, 1 + h, grid[d].get(h, 0))
    if grid:
        ws.conditional_format(2, 1, 1 + len(grid), 24, {"type": "2_color_scale", "min_color": "#FFFFFF",
                                                         "max_color": "#F8696B"})
    ws.set_column(0, 0, 12)
    ws.set_column(1, 24, 4)

    # ---------------------------------------------------------- Config
    ws = wb.add_worksheet("Config")
    ws.set_column(0, 0, 28)
    ws.set_column(1, 1, 120, F["wrap"])
    for i, (k, v) in enumerate(cfg.to_dict().items()):
        ws.write(i, 0, k, F["b"])
        ws.write(i, 1, json.dumps(v) if isinstance(v, (list, dict)) else str(v))
    wb.close()
    log.info("Excel report: %s (%d events, %d thumbnails)", path, len(events), thumbs_used[0])


# ====================================================================== HTML
def build_html(store, cfg, path, max_embedded=1500):
    out = cfg.output_dir
    events, videos, tracks, faces, tamper = load(store)
    idt = identities_table(tracks)
    by_score = sorted(events, key=lambda e: -(e["score"] or 0))
    embed_ids = {e["event_id"] for e in by_score[:max_embedded]}

    def b64(src, w=360):
        t = _thumb(src, out, w)
        if not t:
            return ""
        with open(t, "rb") as fh:
            return "data:image/jpeg;base64," + base64.b64encode(fh.read()).decode()

    cards = []
    for e in events:
        snaps = e["snapshots"] or []
        best = max(snaps, key=lambda s: s.get("persons", 0)) if snaps else {}
        img = b64(best.get("annotated", "")) if e["event_id"] in embed_ids else ""
        flags = "".join(f"<li><b>{html.escape(f['flag'])}</b> +{f['points']:g}: {html.escape(f['reason'])}</li>"
                        for f in e["flags"])
        vlm = html.escape(_vlm_text(e["vlm"]))
        links = [f'<a href="{html.escape(_rel(e["contact_sheet"], out))}" target="_blank">contact sheet</a>']
        if e["clip_path"]:
            links.append(f'<a href="{html.escape(_rel(e["clip_path"], out))}" target="_blank">clip</a>')
        links.append(f'<a href="{html.escape(_rel(e["folder"], out))}" target="_blank">folder</a>')
        objs = ", ".join(f"{k}×{n}" for k, n in (e["objects"] or {}).items())
        cards.append(f"""
<div class="card" data-level="{e['level']}" data-cam="{html.escape(e['camera'] or '')}" data-date="{(e['start_dt'] or '')[:10]}"
 data-score="{e['score']}" data-people="{e['n_person_tracks']}" data-text="{html.escape((e['event_id'] + ' ' + objs + ' ' + vlm + ' ' + ' '.join(e['identities'] or [])).lower())}">
 <div class="hd"><span class="lv {e['level']}">{e['level']} {e['score']:g}</span> <b>{html.escape(e['event_id'])}</b></div>
 {'<a href="' + html.escape(_rel(best.get('annotated', ''), out)) + '" target="_blank"><img loading="lazy" src="' + img + '"></a>' if img else ''}
 <div class="meta">🕒 {e['start_dt'] or '?'} → {(e['end_dt'] or '')[11:]} ({e['duration']:.0f}s)<br>
 🎞 {html.escape(e['rel_path'])} @ <b>{V.fmt_offset(e['start_off'])}</b>–{V.fmt_offset(e['end_off'])}<br>
 👥 {e['max_persons']} max · {e['n_person_tracks']} tracks · faces {e['faces_detected']} {html.escape(', '.join(e['identities'] or []))}<br>
 📦 {html.escape(objs)}</div>
 {'<div class="vlm">🤖 ' + vlm + '</div>' if vlm else ''}
 {'<ul class="flags">' + flags + '</ul>' if flags else ''}
 <div class="links">{' · '.join(links)}</div></div>""")
    cams = sorted({e["camera"] for e in events})
    idcards = "".join(
        f'<div class="id"><img src="{b64(r_["sample"], 120)}"><br><b>{pid}</b> {html.escape(r_["name"])}'
        f'<br>{len(r_["events"])} events</div>' for pid, r_ in list(idt.items())[:200] if r_["sample"])
    tot = sum(v["duration"] or 0 for v in videos)
    mot = sum(v["motion_seconds"] or 0 for v in videos)
    lv = Counter(e["level"] for e in events)
    doc = f"""<!doctype html><html><head><meta charset="utf-8"><title>CCTV Analysis Report</title>
<meta name="viewport" content="width=device-width,initial-scale=1">
<style>body{{font-family:system-ui,Segoe UI,Arial;margin:0;background:#f4f6fa;color:#222}}
header{{background:#1f3864;color:#fff;padding:14px 20px}} h1{{margin:0;font-size:22px}}
.kpi{{display:flex;flex-wrap:wrap;gap:10px;padding:12px 20px}} .kpi div{{background:#fff;border-radius:8px;padding:10px 14px;box-shadow:0 1px 3px #0002}}
.kpi b{{display:block;font-size:20px}} .bar{{padding:6px 20px;display:flex;gap:10px;flex-wrap:wrap;align-items:center;position:sticky;top:0;background:#f4f6faee;z-index:2}}
.grid{{display:grid;grid-template-columns:repeat(auto-fill,minmax(360px,1fr));gap:12px;padding:12px 20px}}
.card{{background:#fff;border-radius:10px;box-shadow:0 1px 4px #0002;padding:10px;font-size:13px}} .card img{{width:100%;border-radius:6px}}
.lv{{padding:2px 8px;border-radius:10px;font-weight:bold;font-size:12px}} .HIGH{{background:{LEVEL_COLORS['HIGH']}}} .REVIEW{{background:{LEVEL_COLORS['REVIEW']}}}
.LOW{{background:{LEVEL_COLORS['LOW']}}} .NORMAL{{background:{LEVEL_COLORS['NORMAL']}}} .meta{{margin:6px 0;line-height:1.5}}
.vlm{{background:#eef3ff;border-radius:6px;padding:6px;margin:4px 0}} .flags{{margin:4px 0;padding-left:18px;color:#a33}}
.ids{{display:flex;flex-wrap:wrap;gap:8px;padding:0 20px 20px}} .id{{background:#fff;border-radius:8px;padding:6px;text-align:center;font-size:12px}} .id img{{width:90px;border-radius:6px}}
h2{{padding:0 20px;font-size:18px}}</style></head><body>
<header><h1>CCTV Analysis Report</h1><div>Generated {dt.datetime.now():%Y-%m-%d %H:%M} · shop hours {cfg.open_hour}–{cfg.close_hour}</div></header>
<div class="kpi"><div><b>{len(videos)}</b>videos</div><div><b>{tot / 3600:.1f} h</b>footage</div>
<div><b>{mot / 3600:.1f} h</b>with motion</div><div><b>{len(events)}</b>events</div>
<div><b>{lv.get('HIGH', 0)}</b>HIGH</div><div><b>{lv.get('REVIEW', 0)}</b>REVIEW</div><div><b>{len(idt)}</b>face IDs</div><div><b>{len(tamper)}</b>tamper</div></div>
<div class="bar">Level <select id="lv"><option value="">all</option><option>HIGH</option><option>REVIEW</option><option>LOW</option><option>NORMAL</option></select>
Camera <select id="cam"><option value="">all</option>{''.join(f'<option>{html.escape(c)}</option>' for c in cams)}</select>
Date <input id="date" type="date"> <label><input id="ppl" type="checkbox"> people only</label>
Search <input id="q" placeholder="person, P0003, bag, theft…"> Sort <select id="sort"><option value="time">time</option><option value="score">score</option></select>
<span id="cnt"></span></div>
<div class="grid" id="grid">{''.join(cards)}</div>
<h2>Face identities</h2><div class="ids">{idcards}</div>
<script>
const g=document.getElementById('grid'),cards=[...g.children];
function f(){{const lv=document.getElementById('lv').value,cam=document.getElementById('cam').value,d=document.getElementById('date').value,
q=document.getElementById('q').value.toLowerCase(),p=document.getElementById('ppl').checked,s=document.getElementById('sort').value;let n=0;
let arr=cards.slice(); if(s=='score') arr.sort((a,b)=>b.dataset.score-a.dataset.score); arr.forEach(c=>g.appendChild(c));
cards.forEach(c=>{{const ok=(!lv||c.dataset.level==lv)&&(!cam||c.dataset.cam==cam)&&(!d||c.dataset.date==d)&&(!p||c.dataset.people>0)&&(!q||c.dataset.text.includes(q));
c.style.display=ok?'':'none'; n+=ok;}}); document.getElementById('cnt').textContent=n+' events';}}
document.querySelectorAll('.bar select,.bar input').forEach(e=>e.addEventListener('input',f)); f();
</script></body></html>"""
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(doc)
    log.info("HTML report: %s", path)


# ====================================================================== charts
def build_charts(store, cfg):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    events = load(store)[0]
    ev = [e for e in events if e["start_dt"]]
    if not ev:
        return []
    os.makedirs(os.path.join(cfg.output_dir, "charts"), exist_ok=True)
    paths = []
    cams = sorted({e["camera"] for e in ev})
    days = sorted({e["start_dt"][:10] for e in ev})[-31:]
    fig, ax = plt.subplots(figsize=(14, max(3, 0.45 * len(days) * max(1, len(cams)) + 1)))
    ylabels = []
    y = 0
    colors = {"HIGH": "#d62728", "REVIEW": "#ff7f0e", "LOW": "#bcbd22", "NORMAL": "#2ca02c"}
    for d in days:
        for c in cams:
            for e in ev:
                if e["start_dt"][:10] == d and e["camera"] == c:
                    h0 = int(e["start_dt"][11:13]) + int(e["start_dt"][14:16]) / 60 + int(e["start_dt"][17:19]) / 3600
                    ax.barh(y, max(e["duration"] / 3600, 0.01), left=h0, color=colors.get(e["level"], "grey"), height=0.7)
            ylabels.append(f"{d} {c}")
            y += 1
    ax.set_yticks(range(len(ylabels)))
    ax.set_yticklabels(ylabels, fontsize=8)
    ax.set_xlim(0, 24)
    ax.set_xticks(range(0, 25, 2))
    ax.set_xlabel("hour of day")
    o = [int(x) for x in cfg.open_hour.split(":")]
    c_ = [int(x) for x in cfg.close_hour.split(":")]
    ax.axvspan(0, o[0] + o[1] / 60, color="#000", alpha=0.06)
    ax.axvspan(c_[0] + c_[1] / 60, 24, color="#000", alpha=0.06)
    ax.set_title("Event timeline (colour = level, shaded = closed hours)")
    ax.invert_yaxis()
    fig.tight_layout()
    p = os.path.join(cfg.output_dir, "charts", "timeline.png")
    fig.savefig(p, dpi=110)
    plt.close(fig)
    paths.append(p)
    fig, ax = plt.subplots(figsize=(10, 3.5))
    hours = Counter(int(e["start_dt"][11:13]) for e in ev if e["n_person_tracks"])
    ax.bar(range(24), [hours.get(h, 0) for h in range(24)], color="#1f77b4")
    ax.set_xticks(range(24))
    ax.set_title("Events with people by hour of day")
    fig.tight_layout()
    p = os.path.join(cfg.output_dir, "charts", "people_by_hour.png")
    fig.savefig(p, dpi=110)
    plt.close(fig)
    paths.append(p)
    return paths


# ================================================================ highlight reel
def build_highlights(store, cfg) -> str:
    ff = V.FFMPEG
    if not ff:
        return ""
    events = [e for e in load(store)[0] if e["clip_path"] and os.path.exists(e["clip_path"])
              and e["level"] in ("HIGH", "REVIEW")]
    events = sorted(events, key=lambda e: -(e["score"] or 0))[: cfg.highlight_max_events]
    events.sort(key=lambda e: e["start_dt"] or "")
    if not events:
        return ""
    tmp = os.path.join(cfg.work_dir, "hl")
    os.makedirs(tmp, exist_ok=True)
    parts = []
    for i, e in enumerate(events):
        p = os.path.join(tmp, f"{i:04d}.mp4")
        txt = f"{e['level']} {e['score']:g} {e['camera']} {e['start_dt'] or ''}".replace(":", r"\:")
        vf = ("scale=960:540:force_original_aspect_ratio=decrease,pad=960:540:(ow-iw)/2:(oh-ih)/2,fps=15,"
              f"drawtext=text='{txt}':x=10:y=10:fontsize=22:fontcolor=yellow:box=1:boxcolor=black@0.6")
        cmd = [ff, "-nostdin", "-y", "-v", "error", "-i", e["clip_path"], "-t", "30", "-vf", vf, "-an",
               "-c:v", "libx264", "-preset", "veryfast", "-crf", "28", "-pix_fmt", "yuv420p", p]
        r = subprocess.run(cmd, capture_output=True)
        if r.returncode != 0:      # drawtext unavailable -> plain
            cmd[cmd.index("-vf") + 1] = vf.split(",drawtext")[0]
            r = subprocess.run(cmd, capture_output=True)
        if r.returncode == 0:
            parts.append(p)
    if not parts:
        return ""
    lst = os.path.join(tmp, "list.txt")
    with open(lst, "w") as fh:
        fh.writelines(f"file '{p}'\n" for p in parts)
    out = os.path.join(cfg.output_dir, "highlight_reel.mp4")
    subprocess.run([ff, "-nostdin", "-y", "-v", "error", "-f", "concat", "-safe", "0", "-i", lst, "-c", "copy",
                    "-movflags", "+faststart", out], capture_output=True)
    for p in parts:
        os.remove(p)
    log.info("Highlight reel: %s (%d events)", out, len(parts))
    return out if os.path.exists(out) else ""


def build_reports(store, cfg) -> dict:
    out = {}
    try:
        out["excel"] = os.path.join(cfg.output_dir, "CCTV_Report.xlsx")
        tmp = os.path.join(cfg.work_dir, "CCTV_Report.xlsx")
        build_excel(store, cfg, tmp)
        import shutil
        try:
            shutil.copyfile(tmp, out["excel"])
        except PermissionError:      # workbook open elsewhere -> timestamped copy
            out["excel"] = out["excel"].replace(".xlsx", f"_{dt.datetime.now():%Y%m%d_%H%M%S}.xlsx")
            shutil.copyfile(tmp, out["excel"])
    except Exception as e:
        log.exception("Excel report failed: %s", e)
    if cfg.make_html:
        try:
            out["html"] = os.path.join(cfg.output_dir, "report.html")
            build_html(store, cfg, out["html"])
        except Exception as e:
            log.exception("HTML report failed: %s", e)
    try:
        out["charts"] = build_charts(store, cfg)
    except Exception as e:
        log.exception("charts failed: %s", e)
    if cfg.make_highlight_reel:
        try:
            out["highlights"] = build_highlights(store, cfg)
        except Exception as e:
            log.exception("highlight reel failed: %s", e)
    return out

In [ ]:
%%writefile /content/cctv_analysis/pipeline.py
"""Orchestrates the whole analysis: discovery -> motion scan -> per-event deep
analysis -> storage -> reports. Safe to re-run: finished videos are skipped."""
from __future__ import annotations

import datetime as dt
import gc
import json
import logging
import os
import re
import time
import traceback
from collections import Counter, defaultdict
from typing import Callable, Optional

import cv2
import numpy as np

from . import annotate as A
from . import rules as R
from . import video_io as V
from .config import Config
from .motion import scan_motion
from .storage import Store

log = logging.getLogger("cctv")
PEOPLE_LIKE = {"person"}
VEHICLES = {"bicycle", "car", "motorcycle", "bus", "truck"}
ANIMALS = {"dog", "cat", "bird", "horse", "cow", "sheep"}


def setup_logging(out_dir: str, verbose: bool = True):
    os.makedirs(out_dir, exist_ok=True)
    log.setLevel(logging.INFO)
    log.handlers.clear()
    fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s", "%H:%M:%S")
    fh = logging.FileHandler(os.path.join(out_dir, "run_log.txt"), encoding="utf-8")
    fh.setFormatter(fmt)
    log.addHandler(fh)
    if verbose:
        sh = logging.StreamHandler()
        sh.setFormatter(fmt)
        log.addHandler(sh)
    log.propagate = False


def _safe(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", s)[:80]


def _iso(d: Optional[dt.datetime]) -> Optional[str]:
    return d.strftime("%Y-%m-%d %H:%M:%S") if d else None


def _imwrite(path: str, img: np.ndarray, q: int = 88) -> str:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    cv2.imwrite(path, img, [cv2.IMWRITE_JPEG_QUALITY, q])
    return path


class Analyzer:
    def __init__(self, cfg: Config, progress: Optional[Callable] = None):
        self.cfg = cfg
        self.progress = progress
        os.makedirs(cfg.output_dir, exist_ok=True)
        os.makedirs(cfg.work_dir, exist_ok=True)
        setup_logging(cfg.output_dir)
        cfg.save(os.path.join(cfg.output_dir, "config_used.json"))
        self.store = Store(os.path.join(cfg.work_dir, "analysis.db"), os.path.join(cfg.output_dir, "analysis.db"))
        from .detector import resolve_device
        V.HW_DECODE = cfg.hw_decode
        self.device = resolve_device(cfg.device)
        log.info("Device: %s", self.device)
        self._det = self._faces = self._sem = self._vlm = None
        self._vlm_failed = False
        self.idb = None
        self.sem_keys, self.sem_embs = [], []
        idx = os.path.join(cfg.output_dir, "semantic_index.npz")
        if os.path.exists(idx):
            try:
                z = np.load(idx, allow_pickle=True)
                self.sem_keys, self.sem_embs = list(z["keys"]), list(z["emb"])
            except Exception:
                pass

    # ------------------------------------------------------------ lazy models
    @property
    def det(self):
        if self._det is None:
            from .detector import Detector
            self._det = Detector(self.cfg)
        return self._det

    @property
    def faces(self):
        if self._faces is None and self.cfg.use_faces:
            from .faces import FaceEngine, IdentityDB
            try:
                self._faces = FaceEngine(self.cfg)
                self.idb = IdentityDB(os.path.join(self.cfg.output_dir, "identities"),
                                      self.cfg.face_match_threshold, self.cfg.known_face_threshold)
                self.idb.enroll_known(self._faces, self.cfg.known_faces_dir)
            except Exception as e:
                log.warning("Face analysis disabled: %s", e)
                self.cfg.use_faces = False
        return self._faces

    @property
    def sem(self):
        if self._sem is None and self.cfg.use_semantic:
            try:
                from .semantic import Semantic
                self._sem = Semantic(self.cfg, self.device)
            except Exception as e:
                log.warning("Semantic tagging disabled: %s", e)
                self.cfg.use_semantic = False
        return self._sem

    @property
    def vlm(self):
        if self._vlm is None and self.cfg.use_vlm and not self._vlm_failed:
            try:
                from .vlm import VLM
                log.info("Loading VLM %s ...", self.cfg.vlm_model)
                self._vlm = VLM(self.cfg, self.device)
            except Exception as e:
                log.warning("VLM disabled: %s", e)
                self._vlm_failed = True
        return self._vlm

    def _osd_reader(self, path, meta):
        if not self.cfg.use_vlm or self.vlm is None:
            return None
        f = V.read_frame_at(path, min(1.0, (meta.get("duration") or 2) / 2), 1280, (meta["width"], meta["height"]))
        return self.vlm.read_clock(f) if f is not None else None

    # ------------------------------------------------------------------- run
    def run(self, build_reports: bool = True) -> dict:
        cfg = self.cfg
        videos = V.find_videos(cfg)
        log.info("Found %d video file(s) under %s", len(videos), cfg.input_dir)
        todo = [p for p in videos if cfg.reprocess or not self.store.video_done(p, V.file_fingerprint(p))]
        log.info("%d to process (%d already done)", len(todo), len(videos) - len(todo))
        t_all = time.time()
        stats = Counter()
        for i, p in enumerate(todo, 1):
            if cfg.max_runtime_minutes and (time.time() - t_all) / 60 >= cfg.max_runtime_minutes:
                log.warning("Time budget of %.0f min reached – %d video(s) left for the next run",
                            cfg.max_runtime_minutes, len(todo) - i + 1)
                stats["videos_deferred"] = len(todo) - i + 1
                break
            log.info("[%d/%d] %s", i, len(todo), p)
            try:
                n = self.process_video(p)
                stats["videos_ok"] += 1
                stats["events"] += n
            except KeyboardInterrupt:
                raise
            except Exception as e:
                stats["videos_failed"] += 1
                log.error("FAILED %s: %s\n%s", p, e, traceback.format_exc())
                try:
                    self.store.db.execute("UPDATE videos SET status='error', error=? WHERE path=?", (str(e)[:2000], p))
                    if not self.store.db.execute("SELECT 1 FROM videos WHERE path=?", (p,)).fetchone():
                        self.store.insert("videos", {"path": p, "rel_path": os.path.relpath(p, cfg.input_dir),
                                                     "status": "error", "error": str(e)[:2000],
                                                     "fingerprint": V.file_fingerprint(p)})
                    self.store.commit()
                except Exception:
                    pass
            if self.progress:
                self.progress(i, len(todo), p)
            gc.collect()
        self._save_side_data()
        self.store.commit()
        log.info("Analysis finished in %.1f min: %s", (time.time() - t_all) / 60, dict(stats))
        if build_reports:
            from .report import build_reports as br
            br(self.store, cfg)
        self.store.commit()
        return dict(stats)

    def _save_side_data(self):
        if self.idb is not None:
            self.idb.save()
        if self.sem_embs:
            np.savez_compressed(os.path.join(self.cfg.output_dir, "semantic_index.npz"),
                                emb=np.stack(self.sem_embs).astype(np.float16), keys=np.array(self.sem_keys, dtype=object))

    # --------------------------------------------------------- single video
    def process_video(self, path: str) -> int:
        cfg = self.cfg
        t0 = time.time()
        meta = V.probe(path)
        if not meta["width"] or not meta["duration"]:
            raise RuntimeError(f"unreadable video (probe: {meta})")
        camera = V.camera_name(path, cfg)
        start_dt, tsrc = V.infer_start_time(path, meta, cfg, vlm_reader=self._osd_reader)
        rel = os.path.relpath(path, cfg.input_dir) if os.path.isdir(cfg.input_dir) else os.path.basename(path)
        vrow = {"path": path, "rel_path": rel, "fingerprint": V.file_fingerprint(path), "camera": camera,
                "start_dt": _iso(start_dt), "time_source": tsrc,
                "end_dt": _iso(start_dt + dt.timedelta(seconds=meta["duration"])) if start_dt else None,
                "duration": meta["duration"], "fps": meta["fps"], "width": meta["width"],
                "height": meta["height"], "codec": meta["codec"], "size_mb": os.path.getsize(path) / 1e6,
                "status": "running", "error": None, "processed_at": dt.datetime.now().isoformat(timespec="seconds"),
                "drive_link": V.drive_link(path) if cfg.drive_links else ""}
        vid = self.store.start_video(vrow)
        log.info("  %s | %dx%d %.1ffps %s | %s | start %s (%s)", camera, meta["width"], meta["height"],
                 meta["fps"], V.fmt_offset(meta["duration"]), meta["codec"], _iso(start_dt), tsrc)

        mr = scan_motion(path, meta, cfg, camera)
        log.info("  motion pass: %d frames, %d event(s), %d tamper, %d lighting change(s) [%.0fs]",
                 mr.frames, len(mr.events), len(mr.tamper), len(mr.lighting_changes), time.time() - t0)
        vdir = os.path.join(cfg.output_dir, "events", _safe(camera or "cam"),
                            start_dt.strftime("%Y-%m-%d") if start_dt else "unknown-date")
        heat_path = ""
        if mr.heatmap is not None and mr.preview is not None:
            heat_path = _imwrite(os.path.join(cfg.output_dir, "heatmaps", _safe(rel) + ".jpg"),
                                 A.heatmap_overlay(mr.preview, mr.heatmap))
        for tp in mr.tamper:
            self.store.insert("tamper", {"video_id": vid, "start_off": tp["start"], "end_off": tp["end"],
                                         "start_dt": _iso(start_dt + dt.timedelta(seconds=tp["start"])) if start_dt else None,
                                         "kind": tp["kind"]}, replace=False)
        # camera tampering or lighting changes outside motion events become their own events
        for kind, ivs in (("tamper", mr.tamper), ("lighting", [{"start": t, "end": t} for t in mr.lighting_changes])):
            for iv in ivs:
                if not any(e["start"] <= iv["end"] and e["end"] >= iv["start"] for e in mr.events):
                    mr.events.append({"start": max(0.0, iv["start"] - 3), "end": min(meta["duration"], iv["end"] + 4),
                                      "motion_peak": 0.0, "motion_mean": 0.0, "motion_active_s": 0.0, "origin": kind})
        mr.events.sort(key=lambda e: e["start"])
        for ev in mr.events:
            ev["lighting_changes"] = sum(1 for t in mr.lighting_changes if ev["start"] <= t <= ev["end"])
        n_saved = 0
        for k, ev in enumerate(mr.events, 1):
            try:
                if self.analyze_event(path, meta, vid, rel, camera, start_dt, ev, k, vdir, mr.tamper):
                    n_saved += 1
            except Exception as e:
                log.error("  event %d failed: %s\n%s", k, e, traceback.format_exc())
            if k % 10 == 0:
                self.store.commit(sync=False)
        motion_s = float(sum(e["end"] - e["start"] for e in mr.events))
        self.store.update_video(vid, status="done", n_events=n_saved, motion_seconds=motion_s,
                                proc_seconds=time.time() - t0, heatmap_path=heat_path,
                                lighting_changes=len(mr.lighting_changes),
                                osd_masked=int(mr.osd_mask is not None),
                                models=json.dumps({"detector": self._det.model_name if self._det else "",
                                                   "pose": self._det.pose_name if self._det else "",
                                                   "semantic": cfg.semantic_model if self._sem else "",
                                                   "vlm": cfg.vlm_model if self._vlm else ""}))
        self._save_side_data()
        self.store.commit()
        log.info("  done: %d event(s) in %.1fs (%.1fx real-time)", n_saved, time.time() - t0,
                 meta["duration"] / max(1e-3, time.time() - t0))
        return n_saved

    # --------------------------------------------------------- single event
    def analyze_event(self, path, meta, vid, rel, camera, start_dt, ev, k, vdir, tamper) -> bool:
        cfg = self.cfg
        s, e = ev["start"], ev["end"]
        dur = max(0.5, e - s)
        ev_dt = start_dt + dt.timedelta(seconds=s) if start_dt else None
        ev["start_dt"] = ev_dt
        eid = f"{_safe(camera or 'cam')}-{ev_dt:%Y%m%d-%H%M%S}-{vid:04d}{k:03d}" if ev_dt else \
            f"{_safe(os.path.splitext(os.path.basename(path))[0])}-{vid:04d}{k:03d}"
        edir = os.path.join(vdir, eid)
        zones = R.zones_for(cfg, camera)
        det = self.det
        det.reset()
        fe = self.faces
        K = max(1, cfg.snapshots_per_event)
        slots = [None] * K
        tracks = {}
        frame_people = []
        det_rows = []
        last_logged = {}
        cls_counter = defaultdict(set)
        untracked = Counter()
        faces_total = 0
        H = W = None
        nframes = 0
        for t, frame in V.iter_frames(path, cfg.detect_fps, cfg.detect_width, s, dur,
                                      src_size=(meta["width"], meta["height"])):
            nframes += 1
            H, W = frame.shape[:2]
            dets = det(frame)
            persons = [d for d in dets if d["cls"] == "person"]
            # ---- faces (only for people still needing good faces)
            ffaces = []
            if fe is not None and persons:
                need = [p for p in persons if p["track"] is None or
                        len(tracks.get(p["track"], {}).get("faces", [])) < cfg.faces_per_track or nframes % 4 == 0]
                ffaces = fe.detect_in_people(frame, need)
                attempted = {p["track"] for p in need}
            else:
                attempted = set()
            faces_total += len(ffaces)
            face_by_track = {f["track"]: f for f in ffaces if f["track"] is not None}
            for d in dets:
                tid = d["track"]
                if tid is None:
                    untracked[d["cls"]] += 1
                    continue
                tr = tracks.setdefault(tid, {"track": tid, "cls": d["cls"], "obs": [], "votes": Counter(),
                                             "zone_frames": Counter(), "hidden_frames": 0, "hands_up": 0,
                                             "crouch": 0, "faces": [], "best_crop": None,
                                             "faces_found": 0})
                tr["votes"][d["cls"]] += d["conf"]
                tr["cls"] = tr["votes"].most_common(1)[0][0]
                tr["obs"].append((t, d["conf"], d["xyxy"]))
                cls_counter[d["cls"]].add(tid)
                x1, y1, x2, y2 = d["xyxy"]
                if d["cls"] == "person":
                    fx, fy = ((x1 + x2) / 2) / W, y2 / H
                    for zn, poly in zones.items():
                        if R.point_in_poly(fx, fy, poly):
                            tr["zone_frames"][zn] += 1
                    from .detector import posture
                    ps = posture(d["kpts"])
                    tr["hands_up"] += ps["hands_up"]
                    tr["crouch"] += ps["crouching"]
                    if fe is not None and tid in attempted and tid not in face_by_track and \
                            ps["face_visible_pose"] and (y2 - y1) > 0.3 * H:
                        tr["hidden_frames"] += 1
                    # best full-body crop: confident, large, not cut by the border
                    edge = x1 < 2 or y1 < 2 or x2 > W - 2 or y2 > H - 2
                    sc = d["conf"] * np.sqrt((x2 - x1) * (y2 - y1)) * (0.6 if edge else 1.0)
                    if tr["best_crop"] is None or sc > tr["best_crop"][0]:
                        pad = 0.06
                        cx1, cy1 = int(max(0, x1 - pad * (x2 - x1))), int(max(0, y1 - pad * (y2 - y1)))
                        cx2, cy2 = int(min(W, x2 + pad * (x2 - x1))), int(min(H, y2 + pad * (y2 - y1)))
                        tr["best_crop"] = (sc, frame[cy1:cy2, cx1:cx2].copy(), t)
                    f = face_by_track.get(tid)
                    if f is not None:
                        tr["faces_found"] += 1
                        q = fe.quality(frame, f)
                        if len(tr["faces"]) < cfg.faces_per_track or q > min(x[0] for x in tr["faces"]):
                            emb = fe.embed(frame, f)
                            tr["faces"].append((q, t, f, emb, fe.crop(frame, f["box"])))
                            tr["faces"].sort(key=lambda x: -x[0])
                            del tr["faces"][cfg.faces_per_track:]
                # sampled detection log: one row per track per second
                if t - last_logged.get(tid, -9) >= 1.0:
                    last_logged[tid] = t
                    det_rows.append({"event_id": eid, "t_off": round(t, 2),
                                     "dt": _iso(start_dt + dt.timedelta(seconds=t)) if start_dt else None,
                                     "cls": d["cls"], "conf": round(d["conf"], 3), "track": tid,
                                     "x1": round(x1 / W, 4), "y1": round(y1 / H, 4),
                                     "x2": round(x2 / W, 4), "y2": round(y2 / H, 4)})
            frame_people.append((t, len(persons)))
            interest = 2.0 * len(persons) + sum(d["conf"] for d in dets) + 0.7 * len(ffaces) + 0.01
            si = min(K - 1, int((t - s) / dur * K))
            if slots[si] is None or interest > slots[si][0]:
                slots[si] = (interest, t, frame.copy(), dets, ffaces)
        if nframes == 0:
            log.warning("  event %s: no frames decoded", eid)
            return False

        # ---------------- track summaries + identities
        fps = cfg.detect_fps
        summaries = []
        id_map = {}
        for tid, tr in tracks.items():
            if len(tr["obs"]) < 2 and max(o[1] for o in tr["obs"]) < 0.6:
                continue                                  # one-off noise
            sm = R.summarize_track(tr, fps, H)
            sm.update({"identity": "", "identity_sim": None, "known_name": "", "face_crop": "",
                       "person_crop": "", "identity_visits": 0})
            if tr["best_crop"] is not None and tr["best_crop"][1].size:
                sm["person_crop"] = _imwrite(os.path.join(edir, "people", f"track{tid:03d}_{tr['cls'].replace(' ', '_')}.jpg"),
                                             tr["best_crop"][1], cfg.snapshot_jpeg_quality)
            embs = [x[3] for x in tr["faces"] if x[3] is not None and x[0] >= cfg.face_min_quality]
            if tr["faces"]:
                q, ft, f, _, crop = tr["faces"][0]
                if crop.size:
                    sm["face_crop"] = _imwrite(os.path.join(edir, "faces", f"track{tid:03d}_best.jpg"), crop, 92)
            if embs and self.idb is not None:
                m = np.mean(embs, 0)
                m /= np.linalg.norm(m) + 1e-9
                when = _iso(start_dt + dt.timedelta(seconds=sm["first"])) if start_dt else rel
                pid, sim, kn = self.idb.assign(m, when or "", sm["face_crop"], eid)
                if sm["face_crop"]:
                    self.idb.set_sample(pid, sm["face_crop"])
                sm.update({"identity": pid, "identity_sim": round(sim, 3), "known_name": kn})
                sm["identity_visits"] = next((len(r.get("events", [])) for r in self.idb.ids if r["id"] == pid), 1)
                id_map[tid] = kn or pid
            for j, (q, ft, f, _, crop) in enumerate(tr["faces"]):
                cp = sm["face_crop"] if j == 0 else (
                    _imwrite(os.path.join(edir, "faces", f"track{tid:03d}_{j}.jpg"), crop, 90) if crop.size else "")
                self.store.insert("faces", {"event_id": eid, "track": tid, "t_off": round(ft, 2),
                                            "dt": _iso(start_dt + dt.timedelta(seconds=ft)) if start_dt else None,
                                            "score": round(f["score"], 3), "quality": round(q, 3),
                                            "box": [round(v, 1) for v in f["box"]], "crop_path": cp,
                                            "identity": sm["identity"], "similarity": sm["identity_sim"],
                                            "known_name": sm["known_name"]}, replace=False)
            summaries.append(sm)

        persons = [x for x in summaries if x["cls"] == "person"]
        objects = {c: len(ids) for c, ids in cls_counter.items()}
        max_persons = max((n for _, n in frame_people), default=0)
        ev["max_persons"] = max_persons
        kinds = set(objects)
        kind = "person" if kinds & PEOPLE_LIKE else "vehicle" if kinds & VEHICLES else \
            "animal" if kinds & ANIMALS else ("object" if kinds else
                                              {"tamper": "camera-tamper", "lighting": "lighting-change"}.get(
                                                  ev.get("origin", ""), "motion-only"))
        if cfg.require_person_for_event and kind in ("motion-only", "object", "lighting-change"):
            return False

        # ---------------- keyframes, semantic tags
        slots = [x for x in slots if x is not None]
        semantic = []
        if self.sem is not None and slots:
            try:
                embs = self.sem.encode_images([x[2] for x in slots])
                semantic = self.sem.classify(embs)
                for x, emb in zip(slots, embs):
                    self.sem_keys.append(f"{eid}|{x[1]:.2f}")
                    self.sem_embs.append(emb.astype(np.float16))
            except Exception as ex:
                log.warning("  semantic failed: %s", ex)
        prelim, _ = R.score_event(ev, summaries, cfg, None, semantic, tamper)

        # ---------------- snapshots
        snaps = []
        sheet_imgs, sheet_caps, raw_imgs = [], [], []
        for j, (_, t, fr, dets, ffs) in enumerate(slots, 1):
            when = start_dt + dt.timedelta(seconds=t) if start_dt else None
            hdr = f"{camera} | {_iso(when) or ''} | {eid} | video {V.fmt_offset(t)}"
            ann = A.draw_dets(fr, dets, ffs, id_map, hdr, zones)
            ap = _imwrite(os.path.join(edir, f"snap{j:02d}_{V.fmt_offset(t).replace(':', '')}_annotated.jpg"),
                          ann, cfg.snapshot_jpeg_quality)
            rp = _imwrite(os.path.join(edir, f"snap{j:02d}_{V.fmt_offset(t).replace(':', '')}_raw.jpg"),
                          fr, cfg.snapshot_jpeg_quality) if cfg.save_raw_snapshots else ""
            npers = sum(1 for d in dets if d["cls"] == "person")
            snaps.append({"t_off": round(t, 2), "dt": _iso(when), "annotated": ap, "raw": rp,
                          "persons": npers, "objects": sorted({d["cls"] for d in dets}),
                          "semantic": semantic[j - 1]["top_label"] if j - 1 < len(semantic) else ""})
            sheet_imgs.append(ann)
            raw_imgs.append(fr)
            sheet_caps.append(f"{j}. {V.fmt_offset(t)}  ({_iso(when)[11:] if when else ''})")
        sheet = A.contact_sheet(sheet_imgs, sheet_caps)
        sheet_path = _imwrite(os.path.join(edir, "contact_sheet.jpg"), sheet, 85)

        # ---------------- VLM description
        vlm_res = None
        want_vlm = cfg.use_vlm and (cfg.vlm_scope == "all" or (cfg.vlm_scope == "people" and persons) or
                                    (cfg.vlm_scope == "flagged" and (prelim >= cfg.suspicion_alert_threshold * 0.5 or
                                                                     (persons and R.is_after_hours(ev_dt, cfg)))))
        vlm_count = getattr(self, "_vlm_count", 0)
        if want_vlm and vlm_count < cfg.vlm_max_events and self.vlm is not None:
            try:
                ctx = f"{len(persons)} person track(s); objects: {', '.join(sorted(objects))}"
                vlm_res = self.vlm.describe_event(A.contact_sheet(raw_imgs, sheet_caps, tile_w=448), ctx)
                self._vlm_count = vlm_count + 1
            except Exception as ex:
                log.warning("  VLM failed: %s", ex)

        score, flags = R.score_event(ev, summaries, cfg, vlm_res, semantic, tamper)
        level = "HIGH" if score >= 70 else "REVIEW" if score >= cfg.suspicion_alert_threshold else \
            "LOW" if score > 0 else "NORMAL"

        # ---------------- clip
        clip = ""
        if cfg.save_clips and (cfg.clip_scope == "all" or (cfg.clip_scope == "people" and (persons or score > 0)) or
                               (cfg.clip_scope == "flagged" and score >= cfg.suspicion_alert_threshold)):
            cp = os.path.join(edir, "clip.mp4")
            lab = f"{camera} {eid} start {_iso(ev_dt) or V.fmt_offset(s)}"
            if V.export_clip(path, s, min(dur, cfg.clip_max_seconds), cp, cfg.clip_width, lab):
                clip = cp

        # ---------------- persist
        for sm in summaries:
            self.store.insert("tracks", {
                "event_id": eid, "track": sm["track"], "cls": sm["cls"], "first_off": round(sm["first"], 2),
                "last_off": round(sm["last"], 2),
                "first_dt": _iso(start_dt + dt.timedelta(seconds=sm["first"])) if start_dt else None,
                "duration": round(sm["duration"], 2), "frames": sm["frames"], "max_conf": round(sm["max_conf"], 3),
                "mean_conf": round(sm["mean_conf"], 3), "max_speed": round(sm["max_speed"], 2),
                "zones": {z: round(v, 1) for z, v in sm["zones"].items()}, "identity": sm["identity"],
                "identity_sim": sm["identity_sim"], "known_name": sm["known_name"], "face_crop": sm["face_crop"],
                "person_crop": sm["person_crop"], "hidden_face_s": round(sm["hidden_face_s"], 1),
                "posture": {"hands_up_s": round(sm["hands_up_s"], 1), "crouch_s": round(sm["crouch_s"], 1)},
                "faces_found": sm["faces_found"], "path": sm["path"]})
        self.store.insert_many("detections", det_rows)
        idents = sorted({(x["known_name"] or x["identity"]) for x in persons if x["identity"]})
        self.store.insert("events", {
            "event_id": eid, "video_id": vid, "video_path": path, "rel_path": rel, "camera": camera,
            "start_off": round(s, 2), "end_off": round(e, 2), "start_dt": _iso(ev_dt),
            "end_dt": _iso(start_dt + dt.timedelta(seconds=e)) if start_dt else None, "duration": round(dur, 2),
            "kind": kind, "motion_peak": round(ev["motion_peak"], 4), "motion_mean": round(ev["motion_mean"], 4),
            "max_persons": max_persons, "n_tracks": len(summaries), "n_person_tracks": len(persons),
            "objects": objects, "faces_detected": faces_total, "identities": idents,
            "best_conf": round(max((x["max_conf"] for x in summaries), default=0.0), 3),
            "score": round(score, 1), "level": level, "flags": flags,
            "semantic": semantic[:K], "vlm": vlm_res, "snapshots": snaps, "contact_sheet": sheet_path,
            "clip_path": clip, "folder": edir, "after_hours": int(R.is_after_hours(ev_dt, cfg)),
            "frames_analyzed": nframes})
        log.info("  event %s %s→%s %s persons=%d tracks=%d faces=%d score=%.0f %s", eid, V.fmt_offset(s),
                 V.fmt_offset(e), kind, len(persons), len(summaries), faces_total, score,
                 ",".join(f["flag"] for f in flags))
        return True


def run(cfg: Config, progress=None, build_reports=True) -> dict:
    return Analyzer(cfg, progress).run(build_reports)

In [ ]:
%%writefile /content/cctv_analysis/helpers.py
"""Notebook helpers: camera/zone preview, in-notebook review of the top
events (images + playable clips), free-text search and search-by-photo."""
from __future__ import annotations

import base64
import json
import os
from collections import defaultdict
from typing import List, Optional

import cv2
import numpy as np

from . import annotate as A
from . import video_io as V
from .config import Config
from .motion import detect_osd_mask, poly_mask
from .rules import zones_for
from .storage import Store


def _show(img_bgr, width=900):
    try:
        from IPython.display import Image, display
        ok, buf = cv2.imencode(".jpg", img_bgr)
        display(Image(data=buf.tobytes(), width=width))
    except Exception:
        pass


def _html(s):
    try:
        from IPython.display import HTML, display
        display(HTML(s))
    except Exception:
        print(s)


def grid_overlay(img):
    h, w = img.shape[:2]
    out = img.copy()
    for i in range(1, 10):
        x, y = int(w * i / 10), int(h * i / 10)
        cv2.line(out, (x, 0), (x, h), (255, 255, 255), 1)
        cv2.line(out, (0, y), (w, y), (255, 255, 255), 1)
        cv2.putText(out, f"{i / 10:.1f}", (x + 2, 14), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 255), 1)
        cv2.putText(out, f"{i / 10:.1f}", (2, y - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 255), 1)
    return out


def preview_cameras(cfg: Config, per_camera: int = 1, width: int = 960):
    """List videos, then show one frame per camera with a 0.1 grid (to write zone
    coordinates), configured zones (red), ignore regions (blue) and the
    auto-detected clock mask (magenta)."""
    vids = V.find_videos(cfg)
    by_cam = defaultdict(list)
    for p in vids:
        by_cam[V.camera_name(p, cfg)].append(p)
    tot = 0.0
    print(f"{len(vids)} video(s) in {len(by_cam)} camera group(s)")
    for cam, ps in by_cam.items():
        print(f"\n=== camera '{cam}': {len(ps)} file(s)")
        for p in ps[:per_camera]:
            meta = V.probe(p)
            tot += meta["duration"]
            st, src = V.infer_start_time(p, meta, cfg)
            print(f"  {os.path.relpath(p, cfg.input_dir)} | {meta['width']}x{meta['height']} {meta['fps']:.1f}fps "
                  f"{V.fmt_offset(meta['duration'])} {meta['codec']} | start {st} ({src})")
            f = V.read_frame_at(p, min(5.0, meta["duration"] / 2), width, (meta["width"], meta["height"]))
            if f is None:
                print("   !! could not decode a frame")
                continue
            img = f.copy()
            z = zones_for(cfg, cam)
            if z:
                A.draw_zones(img, z)
            ign = list(cfg.ignore_regions.get("*", [])) + list(cfg.ignore_regions.get(cam, []))
            if ign:
                m = poly_mask(img.shape[:2], ign)
                img[m > 0] = (img[m > 0] * 0.5 + np.array([255, 0, 0]) * 0.5).astype(np.uint8)
            if cfg.auto_mask_osd:
                osd = detect_osd_mask(p, meta, cfg.motion_width, 30)
                if osd is not None:
                    osd = cv2.resize(osd, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
                    img[osd > 0] = (img[osd > 0] * 0.5 + np.array([255, 0, 255]) * 0.5).astype(np.uint8)
            _show(grid_overlay(img), width)
        for p in ps[per_camera:]:
            tot += V.probe(p)["duration"]
    print(f"\nTotal footage: {tot / 3600:.2f} h")


def _store(cfg):
    return Store(os.path.join(cfg.work_dir, "analysis.db"), os.path.join(cfg.output_dir, "analysis.db"))


def _video_tag(path, width=480):
    if not path or not os.path.exists(path) or os.path.getsize(path) > 40e6:
        return ""
    with open(path, "rb") as fh:
        b = base64.b64encode(fh.read()).decode()
    return f'<video width="{width}" controls preload="none" src="data:video/mp4;base64,{b}"></video>'


def show_events(cfg: Config, where: str = "level IN ('HIGH','REVIEW')", order: str = "score DESC",
                limit: int = 10, clips: bool = True):
    """Display events (contact sheet, reasons, AI description, clip) inside the notebook."""
    st = _store(cfg)
    rows = st.query(f"SELECT * FROM events WHERE {where} ORDER BY {order} LIMIT {int(limit)}")
    if not rows:
        print("No matching events.")
    for e in rows:
        flags = json.loads(e["flags"] or "[]")
        vlm = json.loads(e["vlm"]) if e["vlm"] else None
        _html(f"<h3 style='margin-bottom:2px'>{e['level']} {e['score']:g} — {e['event_id']}</h3>"
              f"<div>🕒 {e['start_dt']} → {e['end_dt']} ({e['duration']:.0f}s) · 🎞 {e['rel_path']} "
              f"@ <b>{V.fmt_offset(e['start_off'])}</b> · 👥 {e['max_persons']} · objects {e['objects']} · "
              f"IDs {e['identities']}</div>"
              + (f"<div style='background:#eef3ff;padding:4px'>🤖 {vlm.get('description', '')} "
                 f"{'⚠️ ' + str(vlm.get('reason', '')) if vlm.get('suspicious') else ''}</div>" if vlm else "")
              + "<ul>" + "".join(f"<li><b>{f['flag']}</b> +{f['points']:g}: {f['reason']}</li>" for f in flags) + "</ul>")
        img = cv2.imread(e["contact_sheet"]) if e["contact_sheet"] else None
        if img is not None:
            _show(img, 1000)
        if clips and e["clip_path"]:
            _html(_video_tag(e["clip_path"]))
    return rows


_SEM_CACHE = {}


def search_text(cfg: Config, query: str, k: int = 12, show: bool = True):
    """Free-text search over all analysed keyframes (SigLIP-2), e.g.
    'person in a red shirt', 'man with a helmet', 'someone carrying a big bag'."""
    from .detector import resolve_device
    from .semantic import Semantic, search_index
    key = (cfg.semantic_model, tuple(cfg.suspicious_prompts), tuple(cfg.normal_prompts))
    sem = _SEM_CACHE.get(key)
    if sem is None:
        sem = _SEM_CACHE[key] = Semantic(cfg, resolve_device(cfg.device))
    hits = search_index(sem, os.path.join(cfg.output_dir, "semantic_index.npz"), query, k)
    st = _store(cfg)
    results = []
    for key, score in hits:
        eid, t = key.split("|")
        e = st.query("SELECT * FROM events WHERE event_id=?", (eid,))
        if not e:
            continue
        e = e[0]
        snaps = json.loads(e["snapshots"] or "[]")
        snap = min(snaps, key=lambda s: abs(s["t_off"] - float(t))) if snaps else None
        results.append({"event_id": eid, "score": score, "time": snap and snap["dt"], "video": e["rel_path"],
                        "offset": V.fmt_offset(float(t)), "snapshot": snap and snap["annotated"]})
    if show:
        for r in results:
            print(f"{r['score']:.3f}  {r['event_id']}  {r['time']}  {r['video']} @ {r['offset']}")
            if r["snapshot"]:
                im = cv2.imread(r["snapshot"])
                if im is not None:
                    _show(im, 640)
    return results


def search_face(cfg: Config, image_path: str, min_sim: Optional[float] = None, show: int = 12):
    """Search-by-photo: find every appearance of the person in `image_path`."""
    from .faces import FaceEngine
    fe = FaceEngine(cfg)
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(image_path)
    faces = sorted(fe.detect_full(img), key=lambda f: -(f["box"][2] - f["box"][0]))
    if not faces:
        print("No face found in the photo.")
        return []
    q = fe.embed(img, faces[0])
    path = os.path.join(cfg.output_dir, "identities")
    ids = json.load(open(path + ".json"))
    emb = np.load(path + ".npy")
    sims = emb @ q
    thr = cfg.face_match_threshold - 0.05 if min_sim is None else min_sim
    order = [i for i in np.argsort(-sims) if sims[i] >= thr]
    st = _store(cfg)
    out = []
    for i in order:
        pid = ids[i]["id"]
        rows = st.query("SELECT t.event_id, t.first_dt, t.face_crop, e.rel_path, t.first_off, e.camera FROM tracks t "
                        "JOIN events e USING(event_id) WHERE t.identity=? ORDER BY t.first_dt", (pid,))
        print(f"\n{pid} similarity {sims[i]:.3f} — {len(rows)} appearance(s)")
        for r in rows:
            print(f"   {r['first_dt']}  {r['camera']}  {r['event_id']}  {r['rel_path']} @ {V.fmt_offset(r['first_off'])}")
            out.append({**r, "identity": pid, "similarity": float(sims[i])})
        if rows and rows[0]["face_crop"] and show:
            im = cv2.imread(rows[0]["face_crop"])
            if im is not None:
                _show(im, 120)
    if not out:
        print("No matching identity found.")
    return out


def summary(cfg: Config):
    st = _store(cfg)
    q = lambda s: st.query(s)[0]
    v = q("SELECT COUNT(*) n, SUM(duration) d, SUM(motion_seconds) m FROM videos WHERE status='done'")
    e = st.query("SELECT level, COUNT(*) n FROM events GROUP BY level")
    print(f"Videos: {v['n']}  footage: {(v['d'] or 0) / 3600:.2f} h  motion: {(v['m'] or 0) / 3600:.2f} h")
    print("Events by level:", {r["level"]: r["n"] for r in e})
    bad = st.query("SELECT rel_path, error FROM videos WHERE status!='done'")
    if bad:
        print("Problems:")
        for b in bad:
            print("  ", b["rel_path"], (b["error"] or "")[:200])

In [ ]:
%%writefile /content/cctv_analysis/__main__.py
"""Command line:  python -m cctv_analysis --input /videos --output /results [--config cfg.json] [key=value ...]"""
import argparse
import json

from .config import Config
from .pipeline import run


def main():
    ap = argparse.ArgumentParser(description="CCTV footage analysis")
    ap.add_argument("--input", required=False)
    ap.add_argument("--output", required=False)
    ap.add_argument("--work", default=None)
    ap.add_argument("--config", default=None, help="JSON file with Config fields")
    ap.add_argument("--reports-only", action="store_true", help="rebuild reports from the database")
    ap.add_argument("overrides", nargs="*", help="field=value (value parsed as JSON when possible)")
    a = ap.parse_args()
    d = {}
    if a.config:
        with open(a.config) as fh:
            d.update(json.load(fh))
    for kv in a.overrides:
        k, v = kv.split("=", 1)
        try:
            d[k] = json.loads(v)
        except json.JSONDecodeError:
            d[k] = v
    if a.input:
        d["input_dir"] = a.input
    if a.output:
        d["output_dir"] = a.output
    if a.work:
        d["work_dir"] = a.work
    cfg = Config.from_dict(d)
    if a.reports_only:
        from .pipeline import Analyzer
        from .report import build_reports
        an = Analyzer(cfg)
        print(build_reports(an.store, cfg))
        an.store.commit()
        return
    print(run(cfg))


if __name__ == "__main__":
    main()

In [ ]:
import sys, importlib, os
os.makedirs('/content/cctv_analysis', exist_ok=True)
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
for m in [k for k in list(sys.modules) if k.startswith('cctv_analysis')]:
    del sys.modules[m]          # pick up edits when re-running
import cctv_analysis
print("cctv_analysis", cctv_analysis.__version__, "ready")

## Step 3 · Settings
Recommended defaults are already filled in. The only must-change items are the two folders.

* **INPUT_DIR** – the Drive folder your DVR/NVR syncs to (sub-folders are scanned; a sub-folder name or `ch01`/`cam2` in the file name becomes the camera name).
* **SPEED_PRESET** – `balanced` is recommended on a T4. `fast` ≈ 2–3× quicker (lighter models, no pose); `accurate` for small faces / far cameras.
* **Shop hours** – activity outside them is flagged as *after hours*.
* **KNOWN_FACES_DIR** *(optional)* – `known_faces/Owner/*.jpg`, `known_faces/Staff_Ravi/*.jpg` … these people get named instead of P-numbers and lower the score of their events.

In [ ]:
#@title Main settings { run: "auto" }
INPUT_DIR = "/content/drive/MyDrive/CCTV"            #@param {type:"string"}
OUTPUT_DIR = "/content/drive/MyDrive/CCTV_Analysis"  #@param {type:"string"}
KNOWN_FACES_DIR = "/content/drive/MyDrive/CCTV_known_faces"  #@param {type:"string"}
SPEED_PRESET = "balanced"  #@param ["fast", "balanced", "accurate"]
TIMEZONE = "Asia/Kolkata"  #@param {type:"string"}
OPEN_HOUR = "09:00"  #@param {type:"string"}
CLOSE_HOUR = "21:30"  #@param {type:"string"}
MOTION_SENSITIVITY = 0.5  #@param {type:"slider", min:0.1, max:0.9, step:0.05}
USE_FACES = True  #@param {type:"boolean"}
USE_AI_DESCRIPTIONS = True  #@param {type:"boolean"}
AI_DESCRIBE = "flagged"  #@param ["flagged", "people", "all"]
SAVE_CLIPS = "people"  #@param ["all", "people", "flagged", "none"]
ONLY_NEWER_THAN_DAYS = 0  #@param {type:"number"}
MAX_VIDEOS = 0  #@param {type:"integer"}
REPROCESS_ALL = False  #@param {type:"boolean"}
STOP_AFTER_MINUTES = 0  #@param {type:"number"}
# STOP_AFTER_MINUTES: e.g. 600 on free Colab so reports are written before the session limit (0 = no limit)

In [ ]:
#@title Advanced settings (zones, ignore areas, file naming) – optional
# Coordinates are fractions of the frame (0..1); use the grid shown in Step 4 to read them.
# "*" applies to every camera; use the camera name (e.g. "ch01") for one camera.
ZONES = {
    # "*":    {"cash_counter": [[0.60, 0.45], [1.00, 0.45], [1.00, 1.00], [0.60, 1.00]]},
    # "ch02": {"stock_room_door": [[0.05, 0.10], [0.25, 0.10], [0.25, 0.80], [0.05, 0.80]]},
}
IGNORE_REGIONS = {
    # "ch01": [[[0.0, 0.0], [0.35, 0.0], [0.35, 0.07], [0.0, 0.07]]],   # e.g. a TV screen / fan / road
}
CLOSED_WEEKDAYS = []          # e.g. [6] = closed on Sunday (0=Mon … 6=Sun)
FILENAME_TIME_PATTERNS = []   # extra regexes with named groups Y m d H M S if your file names are unusual
TIME_OVERRIDES = {}           # {"file.mp4": "2026-10-11 09:30:00"} for files without any time info
EXTRA = {}                    # any other Config field, e.g. {"loiter_seconds": 120, "crowd_size": 5}

In [ ]:
#@title Build configuration
from cctv_analysis.config import Config
PRESETS = {
    "fast":     dict(motion_fps=3, detect_fps=3, detector_model="yolo26n.pt", detector_fallbacks=["yolo11n.pt"],
                     use_pose=False, det_imgsz=640, detect_width=960, semantic_model="google/siglip2-base-patch16-224"),
    "balanced": dict(motion_fps=4, detect_fps=5, detector_model="yolo26s.pt", detector_fallbacks=["yolo11s.pt"],
                     use_pose=True, det_imgsz=960, detect_width=1280, semantic_model="google/siglip2-base-patch16-224"),
    "accurate": dict(motion_fps=5, detect_fps=8, detector_model="yolo26m.pt", detector_fallbacks=["yolo11m.pt"],
                     use_pose=True, pose_model="yolo26s-pose.pt", det_imgsz=1280, detect_width=1920,
                     semantic_model="google/siglip2-so400m-patch14-384", vlm_model="Qwen/Qwen3-VL-4B-Instruct"),
}
cfg = Config(
    input_dir=INPUT_DIR, output_dir=OUTPUT_DIR, work_dir="/content/cctv_work",
    known_faces_dir=KNOWN_FACES_DIR if os.path.isdir(KNOWN_FACES_DIR) else "",
    timezone=TIMEZONE, open_hour=OPEN_HOUR, close_hour=CLOSE_HOUR, closed_weekdays=CLOSED_WEEKDAYS,
    motion_sensitivity=MOTION_SENSITIVITY, use_faces=USE_FACES, use_vlm=USE_AI_DESCRIPTIONS, vlm_scope=AI_DESCRIBE,
    save_clips=SAVE_CLIPS != "none", clip_scope=SAVE_CLIPS if SAVE_CLIPS != "none" else "flagged",
    only_newer_than_days=ONLY_NEWER_THAN_DAYS, max_videos=MAX_VIDEOS, reprocess=REPROCESS_ALL,
    max_runtime_minutes=STOP_AFTER_MINUTES,
    zones=ZONES, ignore_regions=IGNORE_REGIONS, filename_time_patterns=FILENAME_TIME_PATTERNS,
    time_overrides=TIME_OVERRIDES, **PRESETS[SPEED_PRESET])
for k, v in EXTRA.items():
    setattr(cfg, k, v)
assert os.path.isdir(cfg.input_dir), f"INPUT_DIR not found: {cfg.input_dir}"
print("Config ready →", cfg.output_dir)

## Step 4 · Preview (recommended once)
Shows every camera found, how the recording start time was worked out, and one frame per camera with a **0.1 grid** (to write zone coordinates), your **zones** (red), **ignored areas** (blue) and the **auto-detected DVR clock** (magenta, ignored for motion).
If a start time says `file-modified(approx)`, add a pattern or override in *Advanced settings*.

In [ ]:
#@title Preview cameras
from cctv_analysis.helpers import preview_cameras
preview_cameras(cfg)

## Step 5 · Run the analysis
Progress is printed per video and saved after every file, so if Colab disconnects just run this cell again — it continues where it stopped.
Typical speed on a T4 (`balanced`): idle footage ≈ 20–60× real-time, busy footage ≈ 2–6× real-time.

In [ ]:
#@title ▶️ Analyse all videos
from cctv_analysis.pipeline import Analyzer
analyzer = Analyzer(cfg)
stats = analyzer.run(build_reports=True)
print(stats)

## Step 6 · Review results

In [ ]:
#@title Summary + top suspicious events (snapshots, reasons, AI description, playable clip)
from cctv_analysis.helpers import summary, show_events
summary(cfg)
TOP_N = 10  #@param {type:"integer"}
_ = show_events(cfg, where="level IN ('HIGH','REVIEW')", order="score DESC", limit=TOP_N)

In [ ]:
#@title Browse any events (SQL filter)
# Examples:  "after_hours=1"   "camera='ch01' AND start_dt LIKE '2026-10-11%'"   "identities LIKE '%P0003%'"
#            "objects LIKE '%handbag%'"   "start_dt BETWEEN '2026-10-11 20:00' AND '2026-10-11 23:59'"
WHERE = "n_person_tracks > 0"  #@param {type:"string"}
ORDER = "start_dt"  #@param ["start_dt", "score DESC", "duration DESC", "max_persons DESC"]
LIMIT = 10  #@param {type:"integer"}
_ = show_events(cfg, where=WHERE, order=ORDER, limit=LIMIT)

In [ ]:
#@title 🔎 Free-text search over every snapshot
QUERY = "a person wearing a helmet"  #@param {type:"string"}
from cctv_analysis.helpers import search_text
_ = search_text(cfg, QUERY, k=8)

In [ ]:
#@title 🧑 Search by photo – find every appearance of a person
PHOTO = "/content/drive/MyDrive/suspect.jpg"  #@param {type:"string"}
from cctv_analysis.helpers import search_face
_ = search_face(cfg, PHOTO)

In [ ]:
#@title Show outputs (charts, files)
from IPython.display import Image, display
for p in ["charts/timeline.png", "charts/people_by_hour.png"]:
    f = os.path.join(cfg.output_dir, p)
    if os.path.exists(f):
        display(Image(f, width=1000))
print("Excel report :", os.path.join(cfg.output_dir, "CCTV_Report.xlsx"))
print("HTML report  :", os.path.join(cfg.output_dir, "report.html"), "(download the output folder to open it with images)")
print("Highlights   :", os.path.join(cfg.output_dir, "highlight_reel.mp4"))
print("Event folders:", os.path.join(cfg.output_dir, "events"))

In [ ]:
#@title (Optional) Rebuild reports only / download a zip of the reports
from cctv_analysis.report import build_reports
from cctv_analysis.storage import Store
st = Store(os.path.join(cfg.work_dir, "analysis.db"), os.path.join(cfg.output_dir, "analysis.db"))
print(build_reports(st, cfg)); st.commit()
MAKE_ZIP = False  #@param {type:"boolean"}
if MAKE_ZIP:
    import shutil
    z = shutil.make_archive("/content/cctv_report", "zip", cfg.output_dir)
    from google.colab import files; files.download(z)

## Tips & troubleshooting
* **Too many events?** Lower `MOTION_SENSITIVITY`, add `IGNORE_REGIONS` for TVs/fans/roads, or set `EXTRA = {"require_person_for_event": True}`.
* **Missed movement?** Raise `MOTION_SENSITIVITY` (0.6–0.7) or `EXTRA = {"min_motion_area": 0.0008}`.
* **Wrong clock time?** Check Step 4. Supported names include Hikvision `ch01_20261011143000`, Dahua `2026-10-11/…/14.30.00-14.45.00.dav`, `NVR_ch2_main_20261011093015_…`, `11-10-2026_08-00-00`. Otherwise the AI reads the burnt-in DVR clock (when AI descriptions are on), then file time is used.
* **Same person gets several IDs?** Lower `EXTRA = {"face_match_threshold": 0.38}`; **different people merged?** raise it to 0.48.
* **Faster runs:** `SPEED_PRESET="fast"`, `AI_DESCRIBE="flagged"`, `SAVE_CLIPS="flagged"`.
* **`.dav` files** (Dahua) are read natively by ffmpeg; if one fails, export from the DVR as MP4.
* The score is a triage aid: always confirm by watching the cited clip / original video at the given offset.
* Face recognition: use only in line with local privacy law and inform customers that CCTV is in use.